In [ ]:
import sys
sys.path.insert(0, '../../stock_factor_lab_2025/')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 美股實驗彙總

## 回測期間設定

In [ ]:
START_DATE = '2003-3-31'
END_DATE = '2024-12-31'

## 前置作業

### import

In [ ]:
import talib
from get_data import Data
import backtest
from combinations import sim_conditions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
# import plotly.graph_objs as go
import plotly.express as px
from itertools import cycle
from plotly.subplots import make_subplots
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['Microsoft JhengHei']
# plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']  # 微軟正黑體
# plt.rcParams['axes.unicode_minus'] = False  # 用來正常顯示負號
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import FixedLocator, FixedFormatter
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import itertools
import re

from dataframe import CustomDataFrame

In [ ]:
# ind_test = pd.read_csv("../DB_Processor/data/US_data/temp/us_indicator_data_2025-01-01.csv", encoding='utf-8')

In [ ]:
# filtered_df = ind_test[(ind_test['period'].isin(['Q1', 'Q2', 'Q3', 'Q4'])) & 
#                  (ind_test['indicator_name'] == 'eps')]

# eps_pivot_table = filtered_df.pivot(index='date', columns='symbol', values='indicator_value')

In [ ]:
# eps_pivot_table = eps_pivot_table.ffill()["2003":END_DATE]
# eps_pivot_table.index = pd.to_datetime(eps_pivot_table.index)
# eps_pivot_table = CustomDataFrame(eps_pivot_table.resample('ME').last())
# eps_pivot_table

In [ ]:
# no_adj_close_1 = pd.read_csv("../DB_Processor/data/US_data/stock_price_NonAdj_2003-01-01_2024-12-31_A-PDFS.csv", encoding='utf-8')
# no_adj_close_2 = pd.read_csv("../DB_Processor/data/US_data/stock_price_NonAdj_2003-01-01_2024-12-31.csv", encoding='utf-8')

In [ ]:
# # concat no_adj_close_1 and no_adj_close_2
# no_adj_close = pd.concat([no_adj_close_1, no_adj_close_2], ignore_index=True)

In [ ]:
# # to csv
# no_adj_close.to_csv("../2024_code/stock_price_NonAdj_2003-01-01_2024-12-31.csv", index=False, encoding='utf-8-sig')

### get data

In [ ]:
data=Data(market='US')

## 資料下載

In [ ]:
close = data.get('price:close')

In [ ]:
close[START_DATE:END_DATE]

In [ ]:
non_adj_close = pd.read_csv("../2024_code/stock_price_NonAdj_2003-01-01_2024-12-31.csv", encoding="utf-8").pivot(index='date', columns='symbol', values='close')

In [ ]:
non_adj_close.index = pd.to_datetime(non_adj_close.index)
non_adj_close = CustomDataFrame(non_adj_close.ffill())
non_adj_close

### 盈再率

In [ ]:
netIncome = data.get('annual_report_fundamentals:netIncome')

# 4 年加總 #
netIncome_df = netIncome.copy()
# 提取index的月份
netIncome_df['month'] = netIncome_df.index.month
# 依據月份分組，對每個月份的每四年進行加總
netprofit_rol = netIncome_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(4, min_periods=4).sum()) #, include_groups=False)

# 去除稅後淨利為負 (計算盈再率)
adj_netprofit = netprofit_rol[(netprofit_rol > 0) & (netIncome > 0)]

# 長期投資
longTermInvestments = data.get('annual_report_fundamentals:longTermInvestments')
# 固定資產
propertyPlantEquipmentNet = data.get('annual_report_fundamentals:propertyPlantEquipmentNet')


capex = longTermInvestments + propertyPlantEquipmentNet
# 以月份為單位，所以要減掉 48 個月前的資料 (第四年 - 第0年)
capex_rol = capex.diff(48)


rr = capex_rol / adj_netprofit

### 本益比

In [ ]:
# 每季本益比 #
pe = data.get('quarter_report:PE')
pe = pe[pe > 0] # FMP不會過濾掉負值，但是本益比為負時不能算

In [ ]:
# 計算每日本益比 #
eps = data.get('quarter_report:EPS')
# eps = eps_pivot_table.copy()

# 近四季EPS加總
eps_rol = eps + eps.shift(3) + eps.shift(6) + eps.shift(9)
eps_annual = data.get('annual_report:EPS')[START_DATE:END_DATE]
# pe_daily = close / eps_rol
pe_daily = non_adj_close / eps_rol
pe_daily_test = close / eps_rol
pe_daily = pe_daily[pe_daily > 0]

In [ ]:
# data.get('annual_report:EPS')

### 年度ROE、股利支付率(配息率)、稅前淨利、上市時長

In [ ]:
roe = data.get('annual_report:ROE')
dpr = data.get('annual_report_fundamentals:dividendPayoutRatio')
income_bf_tax = data.get('annual_report_fundamentals:incomeBeforeTax')
comp_profile = data.get('company_profile')

#### 上市櫃滿兩年
在滿兩年之前都設為False

In [ ]:
stock_data = {}

for index, row in comp_profile.iterrows():
    stock_code = row['company_symbol']
    listed_date_ = row['ipo_date']
    end_date = listed_date_ + pd.DateOffset(years=2)
    
    # 創建一個全為 True 的 series
    series = pd.Series(True, index=close.index)
    # 在上市日之前和之後兩年內設置為 False
    series.loc[:end_date] = False
    
    stock_data[stock_code] = series

listed_df = pd.concat(stock_data, axis=1)

## 原始條件

In [ ]:
# ROE 5年平均 > 15%
roe_df = roe.copy()
# 提取index的月份
roe_df['month'] = roe_df.index.month
# 依據月份分組，對每個月份的每5年計算平均
roe_rol = roe_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(5, min_periods=5).mean()) #, include_groups=False)

roe_cond = roe_rol > 0.15

In [ ]:
rr_cond = rr < 0.4

稅後淨利、稅前淨利

In [ ]:
netprofit_cond = netIncome > 75000000
income_bf_tax_cond = income_bf_tax > 75000000

股利支付率 (配息率)

In [ ]:
payout_ratio = dpr[(netIncome > 0) & (dpr > 0)]

# 3 年至少 > 40%
payout_df = payout_ratio.copy()
# 提取index的月份
payout_df['month'] = payout_df.index.month
# 依據月份分組
payout_ratio_rol = payout_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(3, min_periods=3).min()) #, include_groups=False)

payout_cond = payout_ratio_rol > 0.4

上市櫃滿兩年

In [ ]:
listed_cond = listed_df.resample('M').last()

#### 本益比進出場條件

每季本益比

In [ ]:
pe_cond_entry = pe < 12
pe_cond_exit = pe > 30

每月本益比

In [ ]:
daily_pe_entry = (pe_daily < 12).resample('M').last()
daily_pe_entry_test = (pe_daily_test < 12).resample('M').last()
daily_pe_exit = (pe_daily > 30).resample('M').last()

---

### 美股各選股原則滿足條件的公司比例

In [ ]:
stock_data_test = {}

for index, row in comp_profile.iterrows():
    stock_code = row['company_symbol']
    listed_date = row['ipo_date']
    
    # 創建一個全為 True 的 series
    series = pd.Series(True, index=close.index)
    # 在上市日之前和之後兩年內設置為 False
    series.loc[:listed_date] = False
    
    stock_data_test[stock_code] = series

listed_df_count = pd.concat(stock_data_test, axis=1).resample('M').last()

In [ ]:
# fig = plt.figure(figsize=(14, 6))

# # roe_data = (roe_cond[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('ME').last()) * 100
# # rr_data = (rr_cond[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('ME').last()) * 100
# # payout_data = (payout_cond[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('ME').last()) * 100
# # profit_data = (netprofit_cond[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('ME').last()) * 100
# # bf_tax_data = (income_bf_tax_cond[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('ME').last()) * 100
# # listed_data = (listed_cond[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('ME').last()) * 100

# pe_daily_entry_data = (daily_pe_entry[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
# pe_daily_exit_data = (daily_pe_exit[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100

# pe_daily_entry_data_test = (daily_pe_entry_test[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100

# pe_cond_entry_data = (pe_cond_entry[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
# pe_cond_exit_data = (pe_cond_exit[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100

# # # 繪製折線圖
# # plt.plot(roe_data.index, roe_data, marker='o', linestyle='-', markersize=3, label='ROE五年平均 > 15%')
# # plt.plot(rr_data.index, rr_data, marker='s', linestyle='--', markersize=3, label='盈餘再投資率 < 40%')
# # plt.plot(payout_data.index, payout_data, marker='^', linestyle='-.', markersize=3, label='股利支付率三年至少 40%')
# # plt.plot(profit_data.index, profit_data, marker='x', linestyle=':', markersize=3, label='稅後淨利 > 7500萬美元')
# # plt.plot(bf_tax_data.index, bf_tax_data, marker='D', linestyle='-', markersize=3, label='稅前淨利 > 7500萬美元')
# # plt.plot(listed_data.index, listed_data, marker='o', linestyle='-', markersize=1, label='上市櫃滿兩年')

# plt.plot(pe_daily_entry_data.index, pe_daily_entry_data, marker='o', linestyle='-', markersize=3, label='本益比 < 12 (進場條件) 每日本益比')
# plt.plot(pe_daily_entry_data_test.index, pe_daily_entry_data_test, marker='o', linestyle='-', markersize=3, label='本益比 < 12 (進場條件) 每日本益比 (測試)')
# # plt.plot(pe_daily_exit_data.index, pe_daily_exit_data, marker='o', linestyle='-', markersize=3, label='本益比 > 30 (出場條件)')

# # plt.plot(pe_cond_entry_data.index, pe_cond_entry_data, marker='*', linestyle='-', markersize=3, label='本益比 < 12 (進場條件)')
# # plt.plot(pe_cond_exit_data.index, pe_cond_exit_data, marker='*', linestyle='-', markersize=3, label='本益比 > 30 (出場條件)')

# # 設置 X 軸為每年年份
# plt.xlabel("年份", fontsize=16)
# plt.ylabel("占比 (%)", fontsize=16)
# plt.title("2003~2024 美股各選股原則滿足條件占所有上市櫃公司的比例", fontsize=16)
# plt.grid(True)
# plt.xticks(
#     ticks=pd.date_range(start="2003-01-01", end="2024-12-31", freq="YS"),
#     labels=pd.date_range(start="2003-01-01", end="2024-12-31", freq="YS").strftime("%Y"),
#     rotation=45,
#     fontsize=14
# )


# plt.grid(linestyle='--', linewidth=0.5, alpha=0.6)
# plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=14)  # 將圖例移動到圖表右上角外
# plt.tight_layout()

# plt.show()

# # fig.savefig("圖36_美股市場中符合各項選股條件的公司數占所有上市公司數的比例.svg", format='svg', dpi=1200, bbox_inches='tight')

In [ ]:
# fig = plt.figure(figsize=(14, 6))

# pe_daily_entry_data = (daily_pe_entry[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
# pe_daily_exit_data = (daily_pe_exit[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100

# pe_cond_entry_data = (pe_cond_entry[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
# pe_cond_exit_data = (pe_cond_exit[START_DATE:END_DATE].sum(axis=1) / listed_df_count[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100

# plt.plot(pe_daily_entry_data.index, pe_daily_entry_data, marker='o', linestyle='-', markersize=3, label='每日本益比 < 12 進場條件')
# # plt.plot(pe_daily_exit_data.index, pe_daily_exit_data, marker='o', linestyle='-', markersize=3, label='每日本益比 > 30 出場條件', alpha=0.3)

# plt.plot(pe_cond_entry_data.index, pe_cond_entry_data, marker='*', linestyle='-', markersize=3, label='每季本益比 < 12 進場條件')
# # plt.plot(pe_cond_exit_data.index, pe_cond_exit_data, marker='*', linestyle='-', markersize=3, label='每季本益比 > 30 出場條件', alpha=0.3)

# # 設置 X 軸為每年年份
# plt.xlabel("年份", fontsize=16)
# plt.ylabel("占比 (%)", fontsize=16)
# plt.title("2003~2024 美股各選股原則滿足條件占所有上市櫃公司的比例", fontsize=16)
# plt.grid(True)
# plt.xticks(
#     ticks=pd.date_range(start="2003-01-01", end="2024-12-31", freq="YS"),
#     labels=pd.date_range(start="2003-01-01", end="2024-12-31", freq="YS").strftime("%Y"),
#     rotation=45,
#     fontsize=14
# )


# plt.grid(linestyle='--', linewidth=0.5, alpha=0.6)
# plt.legend(loc='upper right', fontsize=14)  # 將圖例移動到圖表右上角外
# plt.tight_layout()

# plt.show()

# # fig.savefig("圖36_美股市場中符合各項選股條件的公司數占所有上市公司數的比例.svg", format='svg', dpi=1200, bbox_inches='tight')

---

## 羅素1000清單

In [ ]:
russell_1000_df = pd.read_csv('./russell_component_lists/russell_1000_company.csv')
russell_1000_symbol = russell_1000_df['Symbol'].to_list()
filtered_russell_1000_symbol = [symbol for symbol in russell_1000_symbol if symbol in close.columns]

## 回測

In [ ]:
# 無本益比進出場 #
# 稅後淨利條件
orig_all_cond = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)[START_DATE:END_DATE]
# 稅前淨利條件
orig_all_cond_inc_bf_tax = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)[START_DATE:END_DATE]

# 有本益比進出場 #
# 每季本益比 #
# 稅後淨利條件
orig_all_cond_and_pe = ((orig_all_cond & pe_cond_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_cond_exit[START_DATE:END_DATE]))
# 稅前淨利條件
orig_all_cond_and_pe_inc_bf_tax = ((orig_all_cond_inc_bf_tax & pe_cond_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond_inc_bf_tax) | pe_cond_exit[START_DATE:END_DATE]))

# 每月月底本益比 #
# 稅後淨利條件
orig_all_cond_and_pe_daily = ((orig_all_cond & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE]))
# 稅前淨利條件
orig_all_cond_and_pe_daily_inc_bf_tax = ((orig_all_cond_inc_bf_tax & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond_inc_bf_tax) | daily_pe_exit[START_DATE:END_DATE]))

In [ ]:
orig_all_cond_and_pe_daily_rep = backtest.sim(orig_all_cond_and_pe_daily, resample='M', data=data)
orig_all_cond_and_pe_daily_rep.display()

In [ ]:
# orig_all_cond_and_pe_daily_rep.create_stacked_returns_plot(5)

In [ ]:
rep_all_cond_dic = {}

rep_all_cond_dic['美股_原始條件_稅後淨利_不含本益比進出場'] = orig_all_cond
rep_all_cond_dic['美股_原始條件_稅前淨利_不含本益比進出場'] = orig_all_cond_inc_bf_tax

rep_all_cond_dic['美股_原始條件_稅後淨利_每季本益比進出場'] = orig_all_cond_and_pe
rep_all_cond_dic['美股_原始條件_稅前淨利_每季本益比進出場'] = orig_all_cond_and_pe_inc_bf_tax

rep_all_cond_dic['美股_原始條件_稅後淨利_每月月底本益比進出場'] = orig_all_cond_and_pe_daily
rep_all_cond_dic['美股_原始條件_稅前淨利_每月月底本益比進出場'] = orig_all_cond_and_pe_daily_inc_bf_tax

In [ ]:
rep_all_cond = sim_conditions(rep_all_cond_dic, resample='M', data=data)

In [ ]:
# rep_all_cond.plot_creturns()

In [ ]:
# rep_all_cond.plot_stats()

In [ ]:
fig = rep_all_cond.plot_reps_stock_counts(['美股_原始條件_稅後淨利_每季本益比進出場', '美股_原始條件_稅後淨利_每月月底本益比進出場'] )
fig.savefig("圖 34美股策略每月月底換股策略入選股數變化比較圖.svg", format='svg', dpi=1200, bbox_inches='tight')

In [ ]:
rep_all_cond.selected_stock_count_analysis()

In [ ]:
# df1, df2 = rep_all_cond.reports["美股_原始條件_稅後淨利_每月月底本益比進出場"].calc_returns_contrib()
# df3, df4 = rep_all_cond.reports["美股_原始條件_稅後淨利_每季本益比進出場"].calc_returns_contrib()

In [ ]:
# df1.to_csv("獲利貢獻_美股_原始條件_稅後淨利_每月月底本益比進出場.csv", encoding='utf-8-sig')
# df3.to_csv("獲利貢獻_美股_原始條件_稅後淨利_每季本益比進出場.csv", encoding='utf-8-sig')

In [ ]:
rep_all_cond_df = rep_all_cond.selected_stock_count_analysis()
rep_all_cond_df = rep_all_cond_df.reset_index()

# 分組資料
no_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('不含本益比進出場')]
monthly_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('月底本益比進出場')]

# 提取x軸標籤
no_pe_labels = [s.split('_')[1:3] for s in no_pe['Strategy']]
no_pe_labels = ['_'.join(label) for label in no_pe_labels]

# 設置圖表
fig = plt.figure(figsize=(12, 6))
width = 0.1

# 設置x軸位置
x = range(len(no_pe_labels))

# 繪製bars
bars1 = plt.bar([i - width*1.5 for i in x], no_pe['CAGR (%)'], width, label='CAGR', color='tab:blue', alpha=0.7)
bars2 = plt.bar([i - width/2 for i in x], no_pe['MDD (%)'], width, label='MDD', color='tab:orange', alpha=0.7)
bars3 = plt.bar([i + width/2 for i in x], monthly_pe['CAGR (%)'], width, label='', color='tab:blue', alpha=0.7)
bars4 = plt.bar([i + width*1.5 for i in x], monthly_pe['MDD (%)'], width, label='', color='tab:orange', alpha=0.7)

# 添加數值標籤
for bar in bars1:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=8)

for bar in bars2:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=8)

for bar in bars3:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=8)

for bar in bars4:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=8)

# 獲取軸對象
ax = plt.gca()

# 設置Y軸刻度，以10為間隔
ax.yaxis.set_major_locator(plt.MultipleLocator(10))

# 設置網格線
plt.grid(True, alpha=0.3, which='both', axis='both')

# 設置主要x軸標籤（策略名稱）
ax.set_xticks([i for i in x])
ax.set_xticklabels(no_pe_labels, rotation=0)

# 添加次要x軸標籤（本益比條件）
ax2 = ax.secondary_xaxis('bottom') 
# 調整標籤位置，使其對齊對應的柱狀圖組
ax2.set_xticks([i - width for i in x] + [i + width for i in x])
ax2.set_xticklabels(['不含本益比進出場']*len(no_pe_labels) + ['每月本益比進出場']*len(no_pe_labels), fontsize=10)

# 移動主要x軸到次要x軸的下方
ax.xaxis.set_label_position('bottom')
ax.xaxis.set_ticks_position('bottom')
ax.spines['bottom'].set_position(('outward', 40))

# 設置圖表外觀
ax.tick_params(axis='y', labelsize=10)

plt.ylabel('百分比', fontsize=12)
plt.title('美股 2003-2024 符合所有條件_比較稅前與稅後淨利_有無本益比進出場_每月換股', fontsize=14)
plt.legend(loc='lower center', ncol=2, fontsize=12)
plt.axhline(0, color='red', linewidth=0.5)

# 調整版面配置
plt.subplots_adjust(bottom=0.2)  # 增加底部空間以容納標籤

# 顯示圖表
plt.show()

fig.savefig('./img/圖 7美股原始策略CAGR與MDD比較圖.svg', dpi=300, bbox_inches='tight')

In [ ]:
# rep_all_cond_df = rep_all_cond.selected_stock_count_analysis()
# rep_all_cond_df = rep_all_cond_df.reset_index()

# # 分組資料
# no_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('不含本益比進出場')]
# monthly_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('月底本益比進出場')]

# # 提取x軸標籤
# no_pe_labels = [s.split('_')[1:3] for s in no_pe['Strategy']]
# no_pe_labels = ['_'.join(label) for label in no_pe_labels]

# # 設置圖表 - 創建兩個子圖
# fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
# width = 0.35

# # 設置x軸位置
# x = range(len(no_pe_labels))

# # 子圖1: CAGR
# bars1 = ax1.bar([i - width/2 for i in x], no_pe['CAGR (%)'], width, label='不含本益比進出場', color='tab:blue', alpha=0.7)
# bars3 = ax1.bar([i + width/2 for i in x], monthly_pe['CAGR (%)'], width, label='月底本益比進出場', color='tab:green', alpha=0.7)

# # 為CAGR添加數值標籤
# for bar in bars1:
#     yval = bar.get_height()
#     ax1.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=9)

# for bar in bars3:
#     yval = bar.get_height()
#     ax1.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=9)

# # 設置CAGR圖的Y軸刻度
# ax1.yaxis.set_major_locator(plt.MultipleLocator(10))
# ax1.grid(True, alpha=0.3, which='both')
# ax1.set_title('CAGR比較 (%)', fontsize=12)
# ax1.set_ylabel('CAGR (%)', fontsize=11)
# ax1.axhline(0, color='red', linewidth=0.5)
# ax1.legend(loc='upper right', fontsize=10)

# # 子圖2: MDD
# bars2 = ax2.bar([i - width/2 for i in x], no_pe['MDD (%)'], width, label='不含本益比進出場', color='tab:orange', alpha=0.7)
# bars4 = ax2.bar([i + width/2 for i in x], monthly_pe['MDD (%)'], width, label='月底本益比進出場', color='tab:red', alpha=0.7)

# # 為MDD添加數值標籤
# for bar in bars2:
#     yval = bar.get_height()
#     ax2.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=9)

# for bar in bars4:
#     yval = bar.get_height()
#     ax2.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 2), ha='center', va='bottom', fontsize=9)

# # 設置MDD圖的Y軸刻度
# ax2.yaxis.set_major_locator(plt.MultipleLocator(10))
# ax2.grid(True, alpha=0.3, which='both')
# ax2.set_title('MDD比較 (%)', fontsize=12)
# ax2.set_ylabel('MDD (%)', fontsize=11)
# ax2.axhline(0, color='red', linewidth=0.5)
# ax2.legend(loc='upper right', fontsize=10)

# # 設置共享的X軸標籤
# ax2.set_xticks([i for i in x])
# ax2.set_xticklabels(no_pe_labels, rotation=15)
# fig.text(0.5, 0.01, '策略', ha='center', fontsize=12)

# # 總標題
# fig.suptitle('美股 2003-2024 符合所有條件_比較稅前與稅後淨利_有無本益比進出場_每月換股', fontsize=14)

# # 調整版面配置
# plt.tight_layout()
# plt.subplots_adjust(bottom=0.1, hspace=0.3)  # 增加子圖間距以及底部空間

# # 顯示圖表
# plt.show()

In [ ]:
# rep_all_cond.selected_stock_count_analysis(ratio=True)

---

In [ ]:
# rep_all_cond.reports['美股_原始條件_稅後淨利_不含本益比進出場'].display()

In [ ]:
# orig_strat_pos, orig_strat_neg = rep_all_cond.reports['美股_原始條件_稅後淨利_不含本益比進出場'].calc_returns_contrib(3)

In [ ]:
# orig_daily_pe_strat_pos, orig_daily_pe_strat_neg = rep_all_cond.reports['美股_原始條件_稅後淨利_每月月底本益比進出場'].calc_returns_contrib(3)

In [ ]:
# _, _ = rep_all_cond.reports['美股_原始條件_稅後淨利_每季本益比進出場'].calc_returns_contrib(5)

In [ ]:
# rep_all_cond.visualize_ytd_performance(['美股_原始條件_稅後淨利_不含本益比進出場', '美股_原始條件_稅後淨利_每月月底本益比進出場'])

In [ ]:
fig = rep_all_cond.plot_reps_stock_counts()
fig.savefig("圖 8美股原始策略入選股數變化圖.svg", format='svg', dpi=1200, bbox_inches='tight')

In [ ]:
# rep_all_cond.reports['美股_原始條件_稅後淨利_每月月底本益比進出場'].plot_company_counts()

## 切分時間段2003~2009、2009~2024

In [ ]:
orig_all_cond_2003_2009 = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)[START_DATE:'2009-3-31']
orig_all_cond_bftax_2003_2009 = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)[START_DATE:'2009-3-31']

orig_all_cond_2009_2024 = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)['2009-3-31':END_DATE]
orig_all_cond_bftax_2009_2024 = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)['2009-3-31':END_DATE]

# all_cond_and_pe_2003_2009 = ((orig_all_cond_2003_2009 & pe_cond_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | pe_cond_exit[START_DATE:'2009-3-31']))
# all_cond_and_pe_bftax_2003_2009 = ((orig_all_cond_bftax_2003_2009 & pe_cond_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_bftax_2003_2009) | pe_cond_exit[START_DATE:'2009-3-31']))

# all_cond_and_pe_2009_2024 = ((orig_all_cond_2009_2024 & pe_cond_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | pe_cond_exit['2009-3-31':END_DATE]))
# all_cond_and_pe_bftax_2009_2024 = ((orig_all_cond_bftax_2009_2024 & pe_cond_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_bftax_2009_2024) | pe_cond_exit['2009-3-31':END_DATE]))

all_cond_and_pe_daily_2003_2009 = ((orig_all_cond_2003_2009 & daily_pe_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | daily_pe_exit[START_DATE:'2009-3-31']))
all_cond_and_pe_daily_bftax_2003_2009 = ((orig_all_cond_bftax_2003_2009 & daily_pe_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_bftax_2003_2009) | daily_pe_exit[START_DATE:'2009-3-31']))

all_cond_and_pe_daily_2009_2024 = ((orig_all_cond_2009_2024 & daily_pe_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | daily_pe_exit['2009-3-31':END_DATE]))
all_cond_and_pe_daily_bftax_2009_2024 = ((orig_all_cond_bftax_2009_2024 & daily_pe_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_bftax_2009_2024) | daily_pe_exit['2009-3-31':END_DATE]))

In [ ]:
time_period_dic = {}

time_period_dic['美股_稅後淨利_不含本益比進出場_2003-2009'] = orig_all_cond_2003_2009
time_period_dic['美股_稅前淨利_不含本益比進出場_2003-2009'] = orig_all_cond_bftax_2003_2009
time_period_dic['美股_稅後淨利_不含本益比進出場_2009-2024'] = orig_all_cond_2009_2024
time_period_dic['美股_稅前淨利_不含本益比進出場_2009-2024'] = orig_all_cond_bftax_2009_2024

# time_period_dic['美股_稅後淨利_每季本益比進出場_2003-2009'] = all_cond_and_pe_2003_2009
# time_period_dic['美股_稅前淨利_每季本益比進出場_2003-2009'] = all_cond_and_pe_bftax_2003_2009
# time_period_dic['美股_稅後淨利_每季本益比進出場_2009-2024'] = all_cond_and_pe_2009_2024
# time_period_dic['美股_稅前淨利_每季本益比進出場_2009-2024'] = all_cond_and_pe_bftax_2009_2024

time_period_dic['美股_稅後淨利_每月月底本益比進出場_2003-2009'] = all_cond_and_pe_daily_2003_2009
time_period_dic['美股_稅前淨利_每月月底本益比進出場_2003-2009'] = all_cond_and_pe_daily_bftax_2003_2009
time_period_dic['美股_稅後淨利_每月月底本益比進出場_2009-2024'] = all_cond_and_pe_daily_2009_2024
time_period_dic['美股_稅前淨利_每月月底本益比進出場_2009-2024'] = all_cond_and_pe_daily_bftax_2009_2024

time_period_rep_collec = sim_conditions(time_period_dic, resample='M', data=data)

In [ ]:
time_period_rep_collec.selected_stock_count_analysis()

In [ ]:
def plot_grouped_bar_chart(df):
    # 重置索引
    df.reset_index(inplace=True)
    
    # 解析 Strategy column
    df[['條件', '策略', '時間段']] = df['Strategy'].str.split('_', expand=True).iloc[:, 1:]

    # 排除非數值列
    numeric_columns = df.select_dtypes(include='number').columns
    grouped = df.groupby(['條件', '策略', '時間段'])[numeric_columns].mean().reset_index()

    # 獲取 unique 的條件和策略
    conditions = grouped['條件'].unique()
    strategies = grouped['策略'].unique()

    # 設置 bar 的寬度和位置
    bar_width = 0.2
    index = range(len(conditions) * len(strategies))

    # 創建子圖
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))

    # 繪製 CAGR (%) 的 grouped bar chart
    for i, time_period in enumerate(['2003-2009', '2009-2024']):
        data = grouped[grouped['時間段'] == time_period]
        cagr_values = data['CAGR (%)'].values
        bar_positions = [x + i * bar_width for x in index]
        ax1.bar(bar_positions, cagr_values, bar_width, label=f'{time_period}')

    # 設置 x 軸標籤
    ax1.set_xticks([x + bar_width / 2 for x in index])
    ax1.set_xticklabels([f'{cond}_{strat}' for cond in conditions for strat in strategies], rotation=0, ha='center', fontsize=14)
    ax1.grid(True, alpha=0.3)

    # 添加標籤和標題
    ax1.set_xlabel('Strategy')
    ax1.set_ylabel('CAGR (%)', fontsize=14)
    ax1.yaxis.set_major_locator(FixedLocator(ax1.get_yticks()))
    ax1.yaxis.set_major_formatter(FixedFormatter([f'{int(x)}' for x in ax1.get_yticks()]))
    ax1.set_title('美股 2009-2009、2009-2024 不同時間段 CAGR 比較', fontsize=18)
    ax1.tick_params(axis='y', labelsize=16)  # 設置Y軸刻度字體大小

    # 繪製 MDD (%) 的 grouped bar chart
    for i, time_period in enumerate(['2003-2009', '2009-2024']):
        data = grouped[grouped['時間段'] == time_period]
        mdd_values = data['MDD (%)'].values
        bar_positions = [x + i * bar_width for x in index]
        ax2.bar(bar_positions, mdd_values, bar_width, label=f'{time_period}')

    # 設置 x 軸標籤
    ax2.set_xticks([x + bar_width / 2 for x in index])
    ax2.set_xticklabels([f'{cond}_{strat}' for cond in conditions for strat in strategies], rotation=0, ha='center', fontsize=14)

    # 添加標籤和標題
    ax2.set_xlabel('Strategy')
    ax2.set_ylabel('MDD (%)', fontsize=14)
    ax2.yaxis.set_major_locator(FixedLocator(ax2.get_yticks()))
    ax2.yaxis.set_major_formatter(FixedFormatter([f'{int(x)}' for x in ax2.get_yticks()]))
    ax2.set_title('美股 2009-2009、2009-2024 不同時間段 MDD 比較', fontsize=18)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='y', labelsize=16)  # 設置Y軸刻度字體大小

    # 添加統一的 legend
    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=18)

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()

    # save plt to svg
    fig.savefig('./img/圖 12美股原始策略CAGR與MDD在兩段時期的差異比較圖.svg', format='svg', bbox_inches='tight')

In [ ]:
time_period_test_df = time_period_rep_collec.selected_stock_count_analysis()
plot_grouped_bar_chart(time_period_test_df)

In [ ]:
us_bench = data.get('ruaindex:close')[START_DATE:END_DATE]
us_bench['close'] = us_bench['close'] / us_bench['close'].iloc[0]

In [ ]:
cum_returns = time_period_rep_collec.reports['美股_稅後淨利_每月月底本益比進出場_2003-2009'].stock_data['cum_returns']
us_close = us_bench['close'].reindex(cum_returns.index)

fig = plt.figure(figsize=(18, 6))

# Plot cumulative returns
plt.plot(cum_returns.index, cum_returns, label='美股_稅後淨利_每月月底本益比進出場_2003-2009_累積報酬')


# Plot normalized close prices
plt.plot(us_close.index, us_close, label='羅素3000指數', color='gray', alpha=0.4)

plt.xlabel('年份')
plt.ylabel('百分比')
plt.axhline(1, color='red', linewidth=0.5)
plt.title('美股_稅後淨利_每月月底本益比進出場_2003-2009_累積報酬', fontsize=16)
plt.legend(fontsize=14)
plt.grid(True, alpha=0.3, linestyle='--')

# Custom y-ticks formatter
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{(x - 1) * 100:.0f}%'))

plt.show()

fig.savefig('./img/圖 14美股原始策略加上進出場條件於2003年至2009年的累積報酬變化圖.svg', format='svg', bbox_inches='tight')

## Rerun

In [ ]:
# time_period_rep_collec.reports['美股_稅後淨利_每月月底本益比進出場_2003-2009'].plot_strategy_cumm_return(title='美股_原始條件_每月月底本益比進出場_2003-2009_累積報酬')

In [ ]:
# cum_returns = rep_all_cond.reports['美股_原始條件_稅後淨利_每月月底本益比進出場'].stock_data['cum_returns']
# us_close = us_bench['close'].reindex(cum_returns.index)

# plt.figure(figsize=(18, 6))

# # Plot cumulative returns
# plt.plot(cum_returns.index, cum_returns, label='美股_原始條件_稅後淨利_每月月底本益比進出場_2003-2024_累積報酬')

# # Plot normalized close prices
# plt.plot(us_close.index, us_close, label='羅素3000指數', color='gray', alpha=0.4)

# plt.xlabel('年份')
# plt.ylabel('百分比')
# plt.axhline(1, color='red', linewidth=0.5)
# plt.title('累積報酬', fontsize=16)
# plt.legend(fontsize=14)
# plt.grid(True, alpha=0.3, linestyle='--')

# # Custom y-ticks formatter
# plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{(x - 1) * 100:.0f}%'))

# plt.show()


In [ ]:
# rep_all_cond.reports['美股_原始條件_稅後淨利_每月月底本益比進出場'].plot_strategy_cumm_return(title='美股_原始條件_每月月底本益比進出場_2003-2024_累積報酬')

## 美股_布林通道濾網

In [ ]:
us_benchmark = data.get('ruaindex:close')['2002':'2024']
upperband, middleband, lowerband = talib.BBANDS(us_benchmark.close, timeperiod=300, nbdevup=2.0, nbdevdn=2.0)

### 建立濾網
1. 第一次跌破下通道先記錄下來，等到下一次跌破而且到更低的收盤價賣出
2. 等回到均線之上再買入

In [ ]:
# 創建一個買賣訊號的DataFrame，初始值全部為True
us_bollinger_signal = pd.Series(True, index=us_benchmark.index)

# 記錄第一次跌破的價格
first_break_price = None
# 記錄是否已經回到lower band之上
crossed_above_lower = False
# 是否進入持續的 False 狀態直到突破中通道
in_selling_state = False

# 遍歷所有的日期
for date in us_benchmark.index:
    price = us_benchmark.close[date]
    lower = lowerband[date]
    middle = middleband[date]
    
    # 第一次跌破下通道
    if price < lower and first_break_price is None:
        first_break_price = price
        crossed_above_lower = False
    
    # 價格回到下通道之上
    elif price > lower:
        crossed_above_lower = True
    
    # 再次跌破下通道且符合賣出條件
    if price < lower and crossed_above_lower and price < first_break_price:

        # print(f'{date} - {price} - {lower} - {first_break_price}')
        
        us_bollinger_signal[date] = False
        in_selling_state = True
    
    # 突破中通道則重置狀態
    if price > middle:
        us_bollinger_signal[date] = True
        first_break_price = None
        crossed_above_lower = False
        in_selling_state = False
    
    # 在突破中通道之前保持賣出狀態
    if in_selling_state and price <= middle:
        us_bollinger_signal[date] = False

In [ ]:
fig = plt.figure(figsize=(20, 6))

plt.plot(upperband['2003-3-31':'2024'],
         label="upperband",color='r',
         linestyle='solid', linewidth=0.5)
plt.plot(middleband['2003-3-31':'2024'],
         label="middleband",color='g',linestyle='--', linewidth=0.5)
plt.plot(lowerband['2003-3-31':'2024'],
         label="lowerband",color='b',
         linestyle='solid', linewidth=0.5)
plt.plot(us_benchmark['2003-3-31':'2024'],
         label="Russell 3000 Index",color='black', linewidth=0.8)

# # 添加背景顏色
# plt.axvspan(pd.Timestamp('2008-2-5'), pd.Timestamp('2009-7-31'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2015-9-28'), pd.Timestamp('2015-10-23'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2020-3-16'), pd.Timestamp('2020-5-26'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2022-5-18'), pd.Timestamp('2023-2-2'), color='gray', alpha=0.25)

spans = [
    ('2008-2-5', '2009-7-31'),
    ('2015-9-28', '2015-10-23'),
    ('2020-3-16', '2020-5-26'),
    ('2022-5-18', '2023-2-2')
]

# 繪製灰底和添加標籤
for start, end in spans:
    # 添加灰底
    plt.axvspan(pd.Timestamp(start), pd.Timestamp(end), color='gray', alpha=0.25)
    
    # 計算區間中點位置(用於放置標籤)
    mid_point = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start))/2
    
    # 添加日期區間標籤
    label_text = f'{start}~{end}'
    plt.text(mid_point, plt.ylim()[1]*1.01, label_text,
             horizontalalignment='center',
             fontsize=10)

plt.title("美股羅素3000指數 布林通道（MA300 標準差=2）", fontsize=16, pad=40) 
plt.xlabel("年份", fontsize=14)
plt.xticks(fontsize=14)
plt.ylabel("Russell 3000 Index")
plt.yticks(fontsize=14)

plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

# fig.savefig('./img/圖 17美股大盤布林通道與出場時間示意圖.svg', format='svg', bbox_inches='tight')

In [ ]:
bolling_filt = orig_all_cond.copy()

aligned_signal = us_bollinger_signal.reindex(bolling_filt.index, method='ffill', fill_value=True)

bolling_filt.loc[aligned_signal.index, :] = aligned_signal.values[:, None]

In [ ]:
filtered_russell_1000_symbol = [symbol for symbol in filtered_russell_1000_symbol if symbol not in ['ALAB', 'LOAR']]

In [ ]:
# END_DATE = '2009-3-31'

In [ ]:
# bf_tax_cond = income_bf_tax > 75000000
orig_cond_bft = rr_cond & roe_cond & income_bf_tax_cond & payout_cond & listed_cond
nodpr_bft_cond = rr_cond & roe_cond & income_bf_tax_cond & listed_cond


overall_russell_filt_conds = {}

# overall_conds['所有條件_盈再率<80_無本益比'] = orig_cond
overall_russell_filt_conds['所有條件_無本益比'] = orig_all_cond[START_DATE:END_DATE]
# overall_conds['所有條件_無本益比_稅前淨利'] = orig_cond_bft[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_羅素1000'] = orig_all_cond[filtered_russell_1000_symbol][START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_羅素1000_布林通道'] = (orig_all_cond[filtered_russell_1000_symbol] & bolling_filt)[START_DATE:END_DATE]
# overall_conds['所有條件_無本益比_稅前淨利_羅素1000'] = orig_cond_bft[START_DATE:END_DATE][filtered_russell_1000_symbol]
overall_russell_filt_conds['所有條件_無本益比_布林通道'] = (orig_all_cond & bolling_filt)[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件'] = (orig_all_cond & (roe > 0.15))[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE] & (roe[START_DATE:END_DATE] > 0.15))[START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件_羅素1000'] = (orig_all_cond & (roe[START_DATE:END_DATE] > 0.15))[filtered_russell_1000_symbol][START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_無本益比_ROE出場條件_羅素1000_布林通道'] = (orig_all_cond & bolling_filt & (roe[START_DATE:END_DATE] > 0.15))[filtered_russell_1000_symbol][START_DATE:END_DATE]

# overall_conds['所有條件_盈再率<80_每季本益比'] = orig_quarter_pe
overall_russell_filt_conds['所有條件_有本益比'] = ((orig_all_cond & daily_pe_entry)[START_DATE:END_DATE]).hold_until(((~orig_all_cond) | daily_pe_exit)[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))
overall_russell_filt_conds['所有條件_有本益比_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件_羅素1000_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])[filtered_russell_1000_symbol]


# overall_conds['所有條件_有本益比_稅前淨利'] = (orig_cond_bft[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_cond_bft[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])
overall_russell_filt_conds['所有條件_有本益比_羅素1000'] = orig_all_cond_and_pe_daily[filtered_russell_1000_symbol][START_DATE:END_DATE]
overall_russell_filt_conds['所有條件_有本益比_羅素1000_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])[filtered_russell_1000_symbol]
overall_russell_filt_conds['所有條件_有本益比_ROE出場條件_羅素1000'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))[filtered_russell_1000_symbol] 
# overall_conds['所有條件_有本益比_稅前淨利_羅素1000'] = (orig_cond_bft[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_cond_bft[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])[filtered_russell_1000_symbol]

# # overall_conds['去掉配息盈再率<80_無本益比'] = test_cond
# overall_conds['去掉配息_無本益比'] = nodpr_cond_opt
# # overall_conds['去掉配息_無本益比_稅前淨利'] = nodpr_bft_cond[START_DATE:END_DATE]
# overall_conds['去掉配息_無本益比_羅素1000'] = nodpr_cond_opt_filter
# # overall_conds['去掉配息_無本益比_稅前淨利_羅素1000'] = nodpr_bft_cond[START_DATE:END_DATE][filtered_russell_1000_symbol]
# # overall_conds['去掉配息_盈再率<80_每季本益比'] = test_noDPR_pe
# overall_conds['去掉配息_無本益比_布林通道'] = nodpr_cond_opt & bolling_1_filt

# overall_conds['去掉配息_有本益比'] = opt_noDPR_pe
# overall_conds['去掉配息_有本益比_ROE出場條件'] = (nodpr_cond_opt & daily_pe_entry[START_DATE:END_DATE]).hold_until((~nodpr_cond_opt) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))
# # overall_conds['去掉配息_有本益比_稅前淨利'] = (nodpr_bft_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~nodpr_bft_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])
# overall_conds['去掉配息_有本益比_布林通道'] = (nodpr_cond_opt & daily_pe_entry[START_DATE:END_DATE]).hold_until((~nodpr_cond_opt) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_1_filt)
# overall_conds['去掉配息_有本益比_布林通道_ROE出場條件'] = (nodpr_cond_opt & daily_pe_entry[START_DATE:END_DATE]).hold_until((~nodpr_cond_opt) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_1_filt)

# overall_conds['去掉配息_有本益比_羅素1000'] = opt_noDPR_pe_filt
# overall_conds['去掉配息_有本益比_羅素1000_ROE出場條件'] = (nodpr_cond_opt_filter & daily_pe_entry[START_DATE:END_DATE]).hold_until((~nodpr_cond_opt_filter) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))
# # overall_conds['去掉配息_有本益比_稅前淨利_羅素1000'] = (nodpr_bft_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~nodpr_bft_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])[filtered_russell_1000_symbol]


# overall_conds['單一條件_ROE五年平均'] = roe_cond[START_DATE:END_DATE]
# # overall_conds['單一條件_ROE五年平均_布林通道'] = roe_cond[START_DATE:END_DATE] & bolling_1_filt
# overall_conds['單一條件_ROE五年平均_羅素1000'] = roe_cond[START_DATE:END_DATE][filtered_russell_1000_symbol]

# overall_conds['單一條件_盈再率<40%'] = rr_cond[START_DATE:END_DATE]
# overall_conds['單一條件_盈再率<40%_羅素1000'] = rr_cond[START_DATE:END_DATE][filtered_russell_1000_symbol]
# # overall_conds['單一條件_盈再率<80%'] = (rr<0.8)[START_DATE:END_DATE]


overall_russell_filt_collecs = sim_conditions(overall_russell_filt_conds, resample='M', data=data)
overall_russell_filt_collecs.selected_stock_count_analysis()

In [ ]:
# overall_russell_filt_collecs.reports["所有條件_有本益比_ROE出場條件_羅素1000_布林通道"].display()

In [ ]:
russell_filt_df = overall_russell_filt_collecs.selected_stock_count_analysis()

In [ ]:
def plot_grouped_bolling_chart(russell_filt_df):

    df = russell_filt_df.copy()
    df.reset_index(inplace=True)
    
    # 創建新的欄位來標記是否包含布林通道
    df['has_bolling'] = df['Strategy'].str.contains('布林通道')
    
    # 獲取基本策略名稱（布林通道之前的部分）
    def get_base_strategy(strategy):
        if '布林通道' in strategy:
            return strategy.split('_布林通道')[0]
        return strategy
    
    df['base_strategy'] = df['Strategy'].apply(get_base_strategy)
    
    # 定義策略順序
    strategy_order = [
        "所有條件_無本益比",
        "所有條件_有本益比",
        "所有條件_無本益比_ROE出場條件",
        "所有條件_有本益比_ROE出場條件",
        "所有條件_無本益比_羅素1000",
        "所有條件_有本益比_羅素1000",
        "所有條件_無本益比_ROE出場條件_羅素1000",
        "所有條件_有本益比_ROE出場條件_羅素1000"
    ]
    
    # 按照指定順序篩選基本策略
    base_strategies = [s for s in strategy_order if s in df['base_strategy'].unique()]
    n_strategies = len(base_strategies)
    
    # 準備繪圖數據
    metrics = ['CAGR (%)', 'MDD (%)']
    
    # 設置圖形大小和樣式
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(max(16, n_strategies * 2), 8))
    bar_width = 0.35
    
    # 設置每組柱狀圖的位置
    indices = range(n_strategies)
    
    # 繪製分組柱狀圖
    for i, (metric, ax) in enumerate(zip(metrics, [ax1, ax2])):
        # 無布林通道的數據
        no_bolling_values = [
            df[(df['base_strategy'] == strategy) & (~df['has_bolling'])][metric].iloc[0]
            if len(df[(df['base_strategy'] == strategy) & (~df['has_bolling'])]) > 0
            else None
            for strategy in base_strategies
        ]
        
        # 有布林通道的數據
        with_bolling_values = [
            df[(df['base_strategy'] == strategy) & (df['has_bolling'])][metric].iloc[0]
            if len(df[(df['base_strategy'] == strategy) & (df['has_bolling'])]) > 0
            else None
            for strategy in base_strategies
        ]
        
        # 繪製柱狀圖
        ax.bar([x - bar_width/2 for x in indices],
               [v for v in no_bolling_values if v is not None],
               bar_width,
               label='無布林通道',
               alpha=0.8)
        
        ax.bar([x + bar_width/2 for x in indices],
               [v for v in with_bolling_values if v is not None],
               bar_width,
               label='有布林通道',
               alpha=0.8)
        
        # 設置圖表樣式
        ax.set_xlabel('策略')
        ax.set_ylabel('百分比 (%)')
        ax.set_title(f'美股_2003~{END_DATE}_{metric} 比較')
        ax.set_xticks(indices)
        ax.set_xticklabels(base_strategies, rotation=45, ha='right', fontsize=14)
        # ax.legend()
        ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 添加共同的legend在底部中央
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, 
            loc='center',
            bbox_to_anchor=(0.5, -0.01),  # 調整legend的位置
            ncol=2,  # 將legend排成兩列
            fontsize=14,
            title_fontsize=16)

    # 調整版面配置
    plt.tight_layout()
    
    plt.show()

    # fig.savefig('./img/圖 22美股2003年至2024年有無加上濾網策略績效比較圖.svg', format='svg', bbox_inches='tight')


plot_grouped_bolling_chart(russell_filt_df)

In [ ]:
# overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_有本益比', '所有條件_有本益比_ROE出場條件', '所有條件_有本益比_羅素1000', '所有條件_有本益比_ROE出場條件_羅素1000', '所有條件_有本益比_布林通道', '所有條件_有本益比_ROE出場條件_布林通道', '所有條件_有本益比_羅素1000_布林通道', '所有條件_有本益比_ROE出場條件_羅素1000_布林通道'])

In [ ]:
# overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_無本益比_布林通道', '所有條件_無本益比_ROE出場條件_布林通道', '所有條件_無本益比_羅素1000_布林通道', '所有條件_無本益比_ROE出場條件_羅素1000_布林通道'])

In [ ]:
# overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_有本益比_布林通道', '所有條件_有本益比_ROE出場條件_布林通道', '所有條件_有本益比_羅素1000_布林通道', '所有條件_有本益比_ROE出場條件_羅素1000_布林通道'])

In [ ]:
fig = overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_無本益比', '所有條件_無本益比_ROE出場條件', '所有條件_無本益比_羅素1000', '所有條件_無本益比_ROE出場條件_羅素1000', '所有條件_有本益比', '所有條件_有本益比_ROE出場條件', '所有條件_有本益比_羅素1000', '所有條件_有本益比_ROE出場條件_羅素1000'])
fig.savefig('./img/圖 23美股策略入選股數變化比較圖（無布林通道濾網）.svg', format='svg', bbox_inches='tight')

In [ ]:
fig = overall_russell_filt_collecs.plot_reps_stock_counts(['所有條件_無本益比_布林通道', '所有條件_無本益比_ROE出場條件_布林通道', '所有條件_無本益比_羅素1000_布林通道', '所有條件_無本益比_ROE出場條件_羅素1000_布林通道', '所有條件_有本益比_布林通道', '所有條件_有本益比_ROE出場條件_布林通道', '所有條件_有本益比_羅素1000_布林通道', '所有條件_有本益比_ROE出場條件_羅素1000_布林通道'])
fig.savefig('./img/圖 24美股策略入選股數變化比較圖（有布林通道濾網）.svg', format='svg', bbox_inches='tight')

In [ ]:
pe_russell_filt_conds = {}

pe_russell_filt_conds['所有條件_有本益比'] = orig_all_cond_and_pe_daily[START_DATE:END_DATE]
# pe_russell_filt_conds['所有條件_有本益比_ROE出場條件'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15))
pe_russell_filt_conds['所有條件_有本益比_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])
# pe_russell_filt_conds['所有條件_有本益比_ROE出場條件_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])
# pe_russell_filt_conds['所有條件_有本益比_ROE出場條件_羅素1000_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 0.15) | ~bolling_filt[START_DATE:END_DATE])[filtered_russell_1000_symbol]

pe_russell_filt_collecs = sim_conditions(pe_russell_filt_conds, resample='M', data=data)

pe_russell_filt_collecs.plot_strategies_cumm_return()

In [ ]:
pe_russell_filt_collecs.plot_strategies_cumm_return().savefig('./img/圖 26 美股策略累積報酬比較圖.svg', format='svg', bbox_inches='tight')

In [ ]:
pe_russell_filt_collecs.plot_strategies_MDD().savefig('./img/圖 28 美股策略 MDD 比較圖.svg', format='svg', bbox_inches='tight')

---

## 不同換股頻率實驗

### 每季本益比

In [ ]:
# 每年底 resample
pe_cond_entry_A = (pe < 12).resample('A').last()[START_DATE:END_DATE]
pe_cond_exit_A = (pe > 30).resample('A').last()[START_DATE:END_DATE]
all_cond_A = orig_all_cond.resample('A').last()[START_DATE:END_DATE]

all_cond_pe_A = (all_cond_A & pe_cond_entry_A).hold_until(~all_cond_A | pe_cond_exit_A)
all_cond_pe_A_rep = backtest.sim(all_cond_pe_A, resample='A', data=data)

# 每年三月底 resample
pe_cond_entry_AMAR = (pe < 12).resample('A-MAR').last()[START_DATE:END_DATE]
pe_cond_exit_AMAR = (pe > 30).resample('A-MAR').last()[START_DATE:END_DATE]
all_cond_AMAR = orig_all_cond.resample('A-MAR').last()[START_DATE:END_DATE]

all_cond_pe_AMAR = (all_cond_AMAR & pe_cond_entry_AMAR).hold_until(~all_cond_AMAR | pe_cond_exit_AMAR)
all_cond_pe_AMAR_rep = backtest.sim(all_cond_pe_AMAR, resample='A-MAR', data=data)

# 三月底 + 15天
all_cond_pe_AMAR15_rep = backtest.sim(all_cond_pe_AMAR, resample='A-MAR', resample_offset='15D', data=data)

# 每季 resample
pe_cond_entry_Q = (pe < 12).resample('Q').last()[START_DATE:END_DATE]
pe_cond_exit_Q = (pe > 30).resample('Q').last()[START_DATE:END_DATE]
all_cond_Q = orig_all_cond.resample('Q').last()[START_DATE:END_DATE]

all_cond_pe_Q = (all_cond_Q & pe_cond_entry_Q).hold_until(~all_cond_Q| pe_cond_exit_Q )
all_cond_pe_Q_rep = backtest.sim(all_cond_pe_Q, resample='Q', data=data)

# 每月月初 resample
pe_cond_entry_MS = (pe < 12).resample('MS').first()[START_DATE:END_DATE]
pe_cond_exit_MS = (pe > 30).resample('MS').first()[START_DATE:END_DATE]
all_cond_MS = orig_all_cond.resample('MS').first()[START_DATE:END_DATE]

all_cond_pe_MS = (all_cond_MS & pe_cond_entry_MS).hold_until(~all_cond_MS | pe_cond_exit_MS)
all_cond_pe_MS_rep = backtest.sim(all_cond_pe_MS, resample='MS', data=data)

all_cond_pe_M15_rep = backtest.sim(all_cond_pe_MS, resample='MS', resample_offset='15D', data=data)

# # 每月月底 resample
# # pe_cond_entry_M = (pe < 12).resample('ME').last()[START_DATE:END_DATE]
# # pe_cond_exit_M = (pe > 30).resample('ME').last()[START_DATE:END_DATE]

# all_cond_pe_M = (orig_all_cond & pe_entry).hold_until(~orig_all_cond | pe_exit)
# all_cond_pe_M_rep = backtest.sim(all_cond_pe_M, resample='ME', data=data)

In [ ]:
# # 每月月初 resample
# pe_cond_entry_MS = (pe < 12).resample('MS').first()[START_DATE:END_DATE]
# pe_cond_exit_MS = (pe > 30).resample('MS').first()[START_DATE:END_DATE]

all_cond_pe_MS_rep = backtest.sim(orig_all_cond_and_pe, resample='MS', data=data)

all_cond_pe_M15_rep = backtest.sim(orig_all_cond_and_pe, resample='MS', resample_offset='15D', data=data)

# # 每月月底 resample
# pe_cond_entry_M = (pe < 12).resample('ME').last()[START_DATE:END_DATE]
# pe_cond_exit_M = (pe > 30).resample('ME').last()[START_DATE:END_DATE]

all_cond_pe_M_rep = backtest.sim(orig_all_cond_and_pe, resample='M', data=data)

# # 每日 resample
# pe_cond_entry_D = (pe < 12).resample('D').last()[START_DATE:END_DATE]
# pe_cond_exit_D = (pe > 30).resample('D').last()[START_DATE:END_DATE]

all_cond_pe_D_rep = backtest.sim(orig_all_cond_and_pe, resample='D', data=data)

In [ ]:
# # 每兩週 resample
# pe_cond_entry_2W = (pe < 12).resample('2W').last()[START_DATE:END_DATE]
# pe_cond_exit_2W = (pe > 30).resample('2W').last()[START_DATE:END_DATE]

all_cond_pe_2W_rep = backtest.sim(orig_all_cond_and_pe, resample='2W', data=data)

# # 每周 resample
# pe_cond_entry_W = (pe < 12).resample('W').last()[START_DATE:END_DATE]
# pe_cond_exit_W = (pe > 30).resample('W').last()[START_DATE:END_DATE]

all_cond_pe_W_rep = backtest.sim(orig_all_cond_and_pe, resample='W', data=data)

In [ ]:
print(f"每年\n{all_cond_pe_A_rep.get_stats()}")
print(f"每年三月底\n{all_cond_pe_AMAR_rep.get_stats()}")
print(f"每季\n{all_cond_pe_Q_rep.get_stats()}")
print(f"每月月初\n{all_cond_pe_MS_rep.get_stats()}")
print(f"每月月中\n{all_cond_pe_M15_rep.get_stats()}")
print(f"每月月底\n{all_cond_pe_M_rep.get_stats()}")
print(f"每兩週\n{all_cond_pe_2W_rep.get_stats()}")
print(f"每周\n{all_cond_pe_W_rep.get_stats()}")
print(f"每日\n{all_cond_pe_D_rep.get_stats()}")

In [ ]:
# orig_all_cond_pe_A_rep.display()

In [ ]:
# orig_all_cond_pe_A_rep.display()

In [ ]:
data_series = {
    '每年年底': all_cond_pe_A_rep.stock_data['company_count'].dropna(),
    '每年三月月底': all_cond_pe_AMAR_rep.stock_data['company_count'].dropna(),
    '每季': all_cond_pe_Q_rep.stock_data['company_count'].dropna(),
    '月初': all_cond_pe_MS_rep.stock_data['company_count'].dropna(),
    '月中': all_cond_pe_M15_rep.stock_data['company_count'].dropna(),
    '月底': all_cond_pe_M_rep.stock_data['company_count'].dropna(),
    '每兩週': all_cond_pe_2W_rep.stock_data['company_count'].dropna(),
    '每週': all_cond_pe_W_rep.stock_data['company_count'].dropna(),
    '每日': all_cond_pe_D_rep.stock_data['company_count']
}

# 定義不同的 linestyle
linestyles = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (3, 1, 1, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2))]
markers = ['o', 's', 'D', '', '', '', '', '', '']

fig = plt.figure(figsize=(20, 8))

for (label, series), linestyle, marker in zip(data_series.items(), linestyles, markers):
    plt.plot(series.index, series.values, linestyle=linestyle, label=label, marker=marker, markersize=2)

# 圖表設定
plt.title('美股_不同換股頻率(resample週期)的入選公司數量變化_每季本益比', fontsize=18)
plt.xlabel('年份', fontsize=14)
plt.ylabel('入選股數', fontsize=16)
plt.legend(fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# 設置 X 軸為每年年份
plt.xticks(
    ticks=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS"),
    labels=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

# 設置 Y 軸為正整數
plt.gca().yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# 顯示圖表
plt.show()

fig.savefig('./img/圖 32美股策略使用每季本益比不同換股週期的入選股數變化.svg', format='svg', bbox_inches='tight')

In [ ]:
print(f"每年年底\n入選股數平均：{round(all_cond_pe_A_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_A_rep.stock_data['company_count'].median()}, min: {all_cond_pe_A_rep.stock_data['company_count'].min()}, max: {all_cond_pe_A_rep.stock_data['company_count'].max()}")

print(f"每年三月底\n入選股數平均：{round(all_cond_pe_AMAR_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_AMAR_rep.stock_data['company_count'].median()}, min: {all_cond_pe_AMAR_rep.stock_data['company_count'].min()}, max: {all_cond_pe_AMAR_rep.stock_data['company_count'].max()}")

print(f"每年三月底 + 15天\n入選股數平均：{round(all_cond_pe_AMAR15_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_AMAR15_rep.stock_data['company_count'].median()}, min: {all_cond_pe_AMAR15_rep.stock_data['company_count'].min()}, max: {all_cond_pe_AMAR15_rep.stock_data['company_count'].max()}")

print(f"每季\n入選股數平均：{round(all_cond_pe_Q_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_Q_rep.stock_data['company_count'].median()}, min: {all_cond_pe_Q_rep.stock_data['company_count'].min()}, max: {all_cond_pe_Q_rep.stock_data['company_count'].max()}")

print(f"每月月初\n入選股數平均：{round(all_cond_pe_MS_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_MS_rep.stock_data['company_count'].median()}, min: {all_cond_pe_MS_rep.stock_data['company_count'].min()}, max: {all_cond_pe_MS_rep.stock_data['company_count'].max()}")

print(f"每月月中\n入選股數平均：{round(all_cond_pe_M15_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_M15_rep.stock_data['company_count'].median()}, min: {all_cond_pe_M15_rep.stock_data['company_count'].min()}, max: {all_cond_pe_M15_rep.stock_data['company_count'].max()}")

print(f"每月月底\n入選股數平均：{round(all_cond_pe_M_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_M_rep.stock_data['company_count'].median()}, min: {all_cond_pe_M_rep.stock_data['company_count'].min()}, max: {all_cond_pe_M_rep.stock_data['company_count'].max()}")

print(f"每兩週\n入選股數平均：{round(all_cond_pe_2W_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_2W_rep.stock_data['company_count'].median()}, min: {all_cond_pe_2W_rep.stock_data['company_count'].min()}, max: {all_cond_pe_2W_rep.stock_data['company_count'].max()}")

print(f"每周\n入選股數平均：{round(all_cond_pe_W_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_W_rep.stock_data['company_count'].median()}, min: {all_cond_pe_W_rep.stock_data['company_count'].min()}, max: {all_cond_pe_W_rep.stock_data['company_count'].max()}")

print(f"每日\n入選股數平均：{round(all_cond_pe_D_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_D_rep.stock_data['company_count'].median()}, min: {all_cond_pe_D_rep.stock_data['company_count'].min()}, max: {all_cond_pe_D_rep.stock_data['company_count'].max()}")

### reports

In [ ]:
# orig_all_cond_pe_A_rep.display()
# orig_all_cond_pe_Q_rep.display()
# orig_all_cond_pe_MS_rep.display()
# orig_all_cond_pe_M15_rep.display()
# orig_all_cond_pe_M_rep.display()
# orig_all_cond_pe_2W_rep.display()
# orig_all_cond_pe_W_rep.display()
# orig_all_cond_pe_D_rep.display()

### 每日本益比

In [ ]:
# 每年底 resample
daily_pe_entry_A = (pe_daily < 12).resample('A').last()[START_DATE:END_DATE]
daily_pe_exit_A = (pe_daily > 30).resample('A').last()[START_DATE:END_DATE]

orig_daily_pe_A = (all_cond_A & daily_pe_entry_A).hold_until(~all_cond_A | daily_pe_exit_A)
orig_daily_pe_A_rep = backtest.sim(orig_daily_pe_A, resample='A', data=data)

# 每年三月底 resample
daily_pe_entry_AMAR = (pe_daily < 12).resample('A-MAR').last()[START_DATE:END_DATE]
daily_pe_exit_AMAR = (pe_daily > 30).resample('A-MAR').last()[START_DATE:END_DATE]

orig_daily_pe_AMAR = (all_cond_AMAR & daily_pe_entry_AMAR).hold_until(~all_cond_AMAR | daily_pe_exit_AMAR)
orig_daily_pe_AMAR_rep = backtest.sim(orig_daily_pe_AMAR, resample='A-MAR', data=data)

# # 每季 resample
daily_pe_entry_Q = (pe_daily < 12).resample('Q').last()[START_DATE:END_DATE]
daily_pe_exit_Q = (pe_daily > 30).resample('Q').last()[START_DATE:END_DATE]

orig_daily_pe_Q = (all_cond_Q & daily_pe_entry_Q).hold_until(~all_cond_Q | daily_pe_exit_Q)
orig_daily_pe_Q_rep = backtest.sim(orig_daily_pe_Q, resample='Q', data=data)

In [ ]:
# 每月月初 resample
daily_pe_entry_MS = (pe_daily < 12).resample('MS').first()[START_DATE:END_DATE]
daily_pe_exit_MS = (pe_daily > 30).resample('MS').first()[START_DATE:END_DATE]

orig_daily_pe_MS = (all_cond_MS & daily_pe_entry_MS).hold_until(~all_cond_MS | daily_pe_exit_MS)
orig_daily_pe_MS_rep = backtest.sim(orig_daily_pe_MS, resample='MS', data=data)

# 每月月中 resample
orig_daily_pe_M15_rep = backtest.sim(orig_daily_pe_MS, resample='MS', resample_offset='15D', data=data)

# 每月月底 resample
orig_daily_pe_M_rep = backtest.sim(orig_all_cond_and_pe_daily, resample='M', data=data)

# 每兩週 resample
daily_pe_entry_2W = (pe_daily < 12).resample('2W').last()[START_DATE:END_DATE]
daily_pe_exit_2W = (pe_daily > 30).resample('2W').last()[START_DATE:END_DATE]

orig_daily_pe_2W = (orig_all_cond & daily_pe_entry_2W).hold_until(~orig_all_cond | daily_pe_exit_2W)
orig_daily_pe_2W_rep = backtest.sim(orig_daily_pe_2W, resample='2W', data=data)

# 每周 resample
daily_pe_entry_W = (pe_daily < 12).resample('W').last()[START_DATE:END_DATE]
daily_pe_exit_W = (pe_daily> 30).resample('W').last()[START_DATE:END_DATE]

orig_daily_pe_W = (orig_all_cond & daily_pe_entry_W).hold_until(~orig_all_cond | daily_pe_exit_W)
orig_daily_pe_W_rep = backtest.sim(orig_daily_pe_W, resample='W', data=data)

In [ ]:
# 每日 resample

orig_daily_pe_D = (orig_all_cond & (pe_daily < 12)).hold_until(~orig_all_cond | (pe_daily> 30))
orig_daily_pe_D_rep = backtest.sim(orig_daily_pe_D, resample='D', data=data)

In [ ]:
# orig_daily_pe_W_rep.plot_company_counts()

In [ ]:
print(f"每年\n{orig_daily_pe_A_rep.get_stats()}")
print(f"每年三月底\n{orig_daily_pe_AMAR_rep.get_stats()}")
print(f"每季\n{orig_daily_pe_Q_rep.get_stats()}")
print(f"每月月初\n{orig_daily_pe_MS_rep.get_stats()}")
print(f"每月月中\n{orig_daily_pe_M15_rep.get_stats()}")
print(f"每月月底\n{orig_daily_pe_M_rep.get_stats()}")
print(f"每兩週\n{orig_daily_pe_2W_rep.get_stats()}")
print(f"每周\n{orig_daily_pe_W_rep.get_stats()}")
print(f"每日\n{orig_daily_pe_D_rep.get_stats()}")

In [ ]:
print(f"每年年底\n入選股數平均:{round(orig_daily_pe_A_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_A_rep.stock_data['company_count'].dropna().median()}")
print(f"每年三月月底\n入選股數平均:{round(orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna().median()}")
print(f"每季\n入選股數平均:{round(orig_daily_pe_Q_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_Q_rep.stock_data['company_count'].dropna().median()}")
print(f"每月月初\n入選股數平均:{round(orig_daily_pe_MS_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_MS_rep.stock_data['company_count'].dropna().median()}")
print(f"每月月中\n入選股數平均:{round(orig_daily_pe_M15_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_M15_rep.stock_data['company_count'].dropna().median()}")
print(f"每月月底\n入選股數平均:{round(orig_daily_pe_M_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_M_rep.stock_data['company_count'].dropna().median()}")
print(f"每兩週\n入選股數平均:{round(orig_daily_pe_2W_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_2W_rep.stock_data['company_count'].dropna().median()}")
print(f"每周\n入選股數平均:{round(orig_daily_pe_W_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_W_rep.stock_data['company_count'].dropna().median()}")
print(f"每日\n入選股數平均:{round(orig_daily_pe_D_rep.stock_data['company_count'].mean(), 2)}, 中位數:{orig_daily_pe_D_rep.stock_data['company_count'].median()}")

In [ ]:
orig_pe_M_stocklist = all_cond_pe_M_rep.trades.reset_index()['stock_id'].to_list()
orig_daily_pe_M_stocklist = orig_daily_pe_M_rep.trades.reset_index()['stock_id'].to_list()
orig_daily_pe_D_stocklist = orig_daily_pe_D_rep.trades.reset_index()['stock_id'].to_list()
orig_pe_D_stocklist = all_cond_pe_D_rep.trades.reset_index()['stock_id'].to_list()
orig_daily_pe_A_stocklist = orig_daily_pe_A_rep.trades.reset_index()['stock_id'].to_list()

# unique_stocks = set(orig_pe_M_stocklist).symmetric_difference(set(orig_daily_pe_M_stocklist))
# unique_stocks_list = list(unique_stocks)
# print(len(unique_stocks_list))

In [ ]:
only_in_daily_pe_M = set(orig_daily_pe_M_stocklist) - set(orig_pe_M_stocklist)
print("Stocks only in 每日本益比每月resample:", len(only_in_daily_pe_M))

only_in_pe_M = set(orig_pe_M_stocklist) - set(orig_daily_pe_M_stocklist)
print("Stocks only in 每季本益比每月resample:", len(only_in_pe_M))

only_in_daily_pe_D = set(orig_daily_pe_D_stocklist) - set(orig_pe_D_stocklist)
print("\n\nStocks only in 每日本益比每日resample:", len(only_in_daily_pe_D))

only_in_pe_D = set(orig_pe_D_stocklist) - set(orig_daily_pe_D_stocklist)
print("Stocks only in 每季本益比每日resample:", len(only_in_pe_D))

In [ ]:
only_in_daily_pe_M = set(orig_daily_pe_M_stocklist) - set(orig_daily_pe_D_stocklist)
print("Stocks only in 每日本益比每月resample:", len(only_in_daily_pe_M))

only_in_daily_pe_D = set(orig_daily_pe_D_stocklist) - set(orig_daily_pe_M_stocklist)
print("Stocks only in 每日本益比每日resample:", len(only_in_daily_pe_D))

In [ ]:
only_in_daily_pe_M = set(orig_daily_pe_M_stocklist) - set(orig_daily_pe_A_stocklist)
print("Stocks only in 每日本益比每月resample:", len(only_in_daily_pe_M))

only_in_daily_pe_A = set(orig_daily_pe_A_stocklist) - set(orig_daily_pe_M_stocklist)
print("Stocks only in 每日本益比每年resample:", len(only_in_daily_pe_A))

In [ ]:
same_stocks = set(orig_pe_M_stocklist).intersection(set(orig_daily_pe_M_stocklist))
same_stocks_list = list(same_stocks)
print(len(same_stocks_list))

In [ ]:
data_series_daily = {
    '每年年底': orig_daily_pe_A_rep.stock_data['company_count'].dropna(),
    # '每年三月月底_Q': orig_all_cond_pe_AMAR_rep.stock_data['company_count'].dropna(),
    '每年三月月底': orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna(),
    '每季': orig_daily_pe_Q_rep.stock_data['company_count'].dropna(),
    '月初': orig_daily_pe_MS_rep.stock_data['company_count'].dropna(),
    '月中': orig_daily_pe_M15_rep.stock_data['company_count'].dropna(),
    # '月底_Q': orig_all_cond_pe_M_rep.stock_data['company_count'].dropna(),
    '月底': orig_daily_pe_M_rep.stock_data['company_count'].dropna(),
    '每兩週': orig_daily_pe_2W_rep.stock_data['company_count'].dropna(),
    '每週': orig_daily_pe_W_rep.stock_data['company_count'].dropna(),
    '每日': orig_daily_pe_D_rep.stock_data['company_count']
}

linestyles = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (3, 1, 1, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2))]
markers = ['o', 's', 'D', '', '', '', '', '', '']

fig = plt.figure(figsize=(20, 8))

for (label, series), linestyle, marker in zip(data_series_daily.items(), linestyles, markers):
    plt.plot(series.index, series.values, linestyle=linestyle, label=label, marker=marker, markersize=2)

# 圖表設定
plt.title('美股_不同換股頻率(resample週期)的入選公司數量變化_每日本益比', fontsize=18)
plt.xlabel('年份', fontsize=14)
plt.ylabel('入選股數', fontsize=16)
plt.legend(fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# 設置 X 軸為每年年份
plt.xticks(
    ticks=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS"),
    labels=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

# 顯示圖表
plt.show()

fig.savefig('圖30_美股原始策略使用每日本益比不同換股週期的入選股數變化.svg', format='svg', bbox_inches='tight')

In [ ]:
data_series_compare = {
    "每季本益比每個月月底resample":all_cond_pe_M_rep.stock_data['company_count'].dropna(), # 每季本益比
    "每日本益比每個月月底resample":orig_daily_pe_M_rep.stock_data['company_count'].dropna(), # 每日本益比
}

plt.figure(figsize=(20, 8))

for (label, series), linestyle, marker in zip(data_series_compare.items(), linestyles, markers):
    plt.plot(series.index, series.values, linestyle=linestyle, label=label, marker=marker, markersize=3)

# 圖表設定
plt.title('美股_每季本益比 vs 每日本益比_每個月月底resample', fontsize=18)
plt.xlabel('年份', fontsize=14)
plt.ylabel('入選股數', fontsize=16)
plt.legend(fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# 設置 X 軸為每年年份
plt.xticks(
    ticks=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS"),
    labels=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

# ylim
plt.ylim(0, 30)
# 設置 Y 軸間隔為10
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(5))

# 顯示圖表
plt.show()

In [ ]:
# 將兩個序列轉換為 DataFrame
df_compare = pd.DataFrame({
    'quarterly_pe': all_cond_pe_M_rep.stock_data['company_count'].dropna(),
    'daily_pe': orig_daily_pe_M_rep.stock_data['company_count'].dropna()
})

# 計算差值 (每日本益比 - 每季本益比)
df_compare['difference'] = df_compare['daily_pe'] - df_compare['quarterly_pe']

# 計算統計值
stats = {
    '最大差距': df_compare['difference'].max(),
    '最小差距': df_compare['difference'].min(),
    '平均差距': df_compare['difference'].mean(),
    '標準差': df_compare['difference'].std()
}

# 印出統計結果
for metric, value in stats.items():
    print(f"{metric}: {value:.2f}")

# 找出最大差距和最小差距發生的日期
max_diff_date = df_compare['difference'].idxmax()
min_diff_date = df_compare['difference'].idxmin()

print(f"\n最大差距發生日期: {max_diff_date.strftime('%Y-%m-%d')}")
print(f"最小差距發生日期: {min_diff_date.strftime('%Y-%m-%d')}")

### reports

In [ ]:
# orig_daily_pe_A_rep.display()
# orig_daily_pe_Q_rep.display()
# orig_daily_pe_MS_rep.display()
# orig_daily_pe_M15_rep.display()
# orig_daily_pe_M_rep.display()
# orig_daily_pe_2W_rep.display()
# orig_daily_pe_W_rep.display()
# orig_daily_pe_D_rep.display()

---

---

## 比較是否考慮某項條件其對績效的影響

In [ ]:
dataframes = {
    'roe_15': roe_cond,
    'rr_cond': rr_cond,
    'payout_ratio_cond': payout_cond,
    'profit_cond': netprofit_cond,
    'listed': listed_cond,
}

# 定義 DataFrame 對應的中文名稱
dataframe_names = {
    'roe_15': 'ROE五年平均',
    'rr_cond': '盈再率',
    'payout_ratio_cond': '配息率',
    'profit_cond': '稅後淨利',
    'hold_cond': '董監持股',
    'listed': '上市櫃滿兩年'
}


# 用於儲存組合結果
compare_conds_strat = {}

# 產生不同長度的組合
counter = 1
for r in range(1, len(dataframes) + 1):
    for combination in itertools.combinations(dataframes.keys(), r):
        # 計算DataFrame間的 AND 運算
        combined_signal = dataframes[combination[0]]
        for df_name in combination[1:]:
            combined_signal &= dataframes[df_name]

        # 限制日期區間
        combined_signal = CustomDataFrame(combined_signal)
        combined_signal = combined_signal[START_DATE:END_DATE]

        # 使用組合的中文名稱生成key_name
        key_name = "+".join([dataframe_names[df] for df in combination])

        # 將計算結果儲存到字典
        compare_conds_strat[key_name] = combined_signal
        counter += 1


compare_strat_collecs = sim_conditions(compare_conds_strat, resample='M', data=data)

In [ ]:
compare_strat_collecs_df = compare_strat_collecs.selected_stock_count_analysis()
# compare_strat_collecs_df.to_csv('./performance_file/US/美股_比較是否考慮某項條件其對績效的影響_無本益比進出場.csv', encoding="cp950")

包含本益比進出場

In [ ]:
# 用於儲存組合結果
compare_conds_strat_PE={}

# 產生不同長度的組合，從1到6
counter = 1
for r in range(1, len(dataframes) + 1):
    for combination in itertools.combinations(dataframes.keys(), r):
        # 計算DataFrame間的 AND 運算
        combined_signal = dataframes[combination[0]]
        for df_name in combination[1:]:
            combined_signal &= dataframes[df_name]

        # 限制日期區間
        combined_signal = CustomDataFrame(combined_signal)
        combined_signal = (combined_signal[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~combined_signal[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])

        # 使用組合的中文名稱生成key_name
        key_name = "+".join([dataframe_names[df] for df in combination])

        # 將計算結果儲存到字典
        compare_conds_strat_PE[key_name] = combined_signal
        counter += 1


compare_strat_collecs_PE = sim_conditions(compare_conds_strat_PE, resample='M', data=data)

In [ ]:
compare_strat_collecs_PE_df = compare_strat_collecs_PE.selected_stock_count_analysis()
# compare_strat_collecs_PE_df.to_csv('./performance_file/US/美股_比較是否考慮某項條件其對績效的影響_本益比進出場.csv', encoding="cp950")

---

In [ ]:
# 條件的字典
condition_dict = {
    'ROE五年平均': roe_cond,
    '盈再率': rr_cond,
    '三年配息率': payout_cond,
    '稅後淨利': netprofit_cond,
    '上市櫃滿兩年': listed_cond,
}

# 分離 "董監持股+上市櫃滿兩年" 與其他條件
additional_condition_key = '上市櫃滿兩年'
additional_condition = condition_dict.pop(additional_condition_key)

# 剩餘條件的 key
keys = list(condition_dict.keys())

# 創建交易訊號字典
signal_dict = {}

# 遍歷 2, 3, 4 的組合長度
for r in range(2, 5):
    combinations = itertools.combinations(keys, r)
    for combo in combinations:
        # 組合名稱
        combo_name = '+'.join(combo)

        # 條件的 AND 結果
        combined_condition = condition_dict[combo[0]]
        for key in combo[1:]:
            combined_condition &= condition_dict[key]

        # 加入不含 "董監持股+上市櫃滿兩年" 的組合
        signal_dict[combo_name] = combined_condition[START_DATE:END_DATE]

        # 加入含 "董監持股+上市櫃滿兩年" 的組合
        combo_name_with_additional = f"{combo_name}_{additional_condition_key}"
        signal_dict[combo_name_with_additional] = (combined_condition & additional_condition)[START_DATE:END_DATE]

# 添加本益比進出場的組合
pe_signal_dict = {}
for name, sig in signal_dict.items():
    # 原組合的本益比進出場
    pe_signal_dict[f"{name}_本益比進出場"] = (sig[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~sig[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])

    # 含 "董監持股+上市櫃滿兩年" 的組合的本益比進出場
    if additional_condition_key in name:
        pe_signal_dict[f"{name}_本益比進出場"] = (sig[START_DATE:END_DATE] & daily_pe_entry[START_DATE:END_DATE]).hold_until((~sig[START_DATE:END_DATE]) | daily_pe_exit[START_DATE:END_DATE])

# 合併所有組合
signal_dict.update(pe_signal_dict)

# 確認結果
for name, condition in signal_dict.items():
    print(name)

In [ ]:
signal_dict_comb = sim_conditions(signal_dict, resample='M', data=data)
signal_dict_comb.selected_stock_count_analysis()
# signal_dict_comb.selected_stock_count_analysis().to_csv('./performance_file/US/美股_ch4.csv', encoding="cp950")

In [ ]:
df = signal_dict_comb.selected_stock_count_analysis()
df.reset_index(inplace=True)

In [ ]:
df

In [ ]:
def create_grouped_bar_chart(df):
    # 從 Strategy 中提取因子組合
    def extract_factor(strategy):
        # 用正則表達式匹配因子部分
        match = re.match(r'^([^_]+(?:\+[^_]+)*).*', strategy)
        if match:
            return match.group(1)
        return strategy
    
    # 為每個 Strategy 添加一個新列 'Factor'，表示其因子組合
    df['Factor'] = df['Strategy'].apply(extract_factor)
    
    # 定義標示類型
    def get_label_type(strategy):
        if '_上市櫃滿兩年_本益比進出場' in strategy:
            return '包含上市櫃滿兩年&本益比進出場'
        elif '_上市櫃滿兩年' in strategy:
            return '包含上市櫃滿兩年'
        elif '_本益比進出場' in strategy:
            return '包含本益比進出場'
        else:
            return '不含上市櫃滿兩年&本益比進出場'
    
    df['Label'] = df['Strategy'].apply(get_label_type)
    
    # 根據圖片中的順序定義因子順序
    factor_order = [
        'ROE五年平均+三年配息率',
        'ROE五年平均+盈再率',
        'ROE五年平均+稅後淨利',
        '盈再率+三年配息率',
        '盈再率+稅後淨利',
        'ROE五年平均+三年配息率+稅後淨利',
        'ROE五年平均+盈再率+三年配息率',
        'ROE五年平均+盈再率+稅後淨利',
        'ROE五年平均+盈再率+三年配息率+稅後淨利'
    ]
    
    # 過濾掉 DataFrame 中沒有的因子
    factor_order = [f for f in factor_order if f in df['Factor'].unique()]
    
    # 獲取唯一的標籤類型
    label_types = ['不含上市櫃滿兩年&本益比進出場', '包含上市櫃滿兩年', '包含本益比進出場', '包含上市櫃滿兩年&本益比進出場']
    colors = ['gray', 'orange', 'tab:blue', 'tab:red']
    
    # 設置圖表大小
    fig = plt.figure(figsize=(14, 8))
    
    # 設置每個組的寬度和組內 bar 的寬度
    n_groups = len(factor_order)
    n_bars = len(label_types)
    group_width = 0.8
    bar_width = group_width / n_bars
    
    # 計算每個 bar 的位置
    indices = np.arange(n_groups)
    
    # 繪製 grouped bar chart
    for i, (label_type, color) in enumerate(zip(label_types, colors)):
        data = []
        for factor in factor_order:
            rows = df[(df['Factor'] == factor) & (df['Label'] == label_type)]
            if not rows.empty:
                data.append(rows['CAGR (%)'].values[0])
            else:
                data.append(0)
        
        positions = indices - group_width/2 + (i + 0.5) * bar_width
        plt.bar(positions, data, bar_width, label=label_type, color=color)
    
    # 設置 x 軸標籤、圖表標題等
    plt.xlabel('Factor')
    plt.ylabel('CAGR (%)', fontsize=16)
    plt.title('美股策略組合CAGR (%)比較')
    plt.xticks(indices, factor_order, rotation=45, ha='right', fontsize=14)
    plt.yticks(fontsize=14)
    # legend 改為橫向
    plt.legend(loc='upper left',  ncol=2, fontsize=13)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 20)  # 設置 y 軸範圍與圖片一致
    
    # 調整佈局
    plt.tight_layout()
    
    # 顯示圖表
    plt.show()

    # 儲存圖表為 SVG 格式
    fig.savefig('./img/圖43 美股策略組合CAGR (%)比較.svg', format='svg', bbox_inches='tight')


# 調用函數繪製圖表
create_grouped_bar_chart(df)

In [ ]:
pd.options.display.float_format = '{:.2f}'.format


def analyze_signal_groups(signal_dict_comb):
    # 初始化結果字典
    results = {
        '包含ROE五年平均': [],
        '包含盈再率': [],
        '包含配息率': [],
        '包含稅後淨利': [],
        # '包含董監持股+上市櫃滿兩年': []
    }

    # 遍歷策略並分類
    for strategy in signal_dict_comb.index:
        if 'ROE五年平均' in strategy:
            results['包含ROE五年平均'].append(strategy)
        if '盈再率' in strategy:
            results['包含盈再率'].append(strategy)
        if '三年配息率' in strategy:
            results['包含配息率'].append(strategy)
        if '稅後淨利' in strategy:
            results['包含稅後淨利'].append(strategy)
        # if '董監持股+上市櫃滿兩年' in strategy:
        #     results['包含董監持股+上市櫃滿兩年'].append(strategy)

    # 計算每組的平均值，細分為包含/不包含條件、有/無本益比進出場、綜合
    summary_data = []
    for group, strategies in results.items():
        condition_name = group.split('包含')[1]

        # 包含條件與不包含條件
        for include_condition in [True, False]:
            filtered_strategies_condition = [s for s in signal_dict_comb.index if (condition_name in s) == include_condition]

            # 細分有/無本益比進出場
            for include_pe in [True, False]:
                filtered_strategies = [s for s in filtered_strategies_condition if ('本益比進出場' in s) == include_pe]
                if filtered_strategies:
                    filtered_df = signal_dict_comb.loc[filtered_strategies]
                    avg_cagr = filtered_df['CAGR (%)'].mean()
                    avg_mdd = filtered_df['MDD (%)'].mean()
                    avg_selected_stock = filtered_df['入選股數平均'].mean()
                    condition_label = f"{'包含' if include_condition else '不包含'}{condition_name} ({'含本益比進出場' if include_pe else '不含本益比進出場'})"
                    summary_data.append([condition_label, avg_cagr, avg_mdd, avg_selected_stock])

            # 綜合分析
            combined_df = signal_dict_comb.loc[filtered_strategies_condition]
            if not combined_df.empty:
                avg_cagr_combined = combined_df['CAGR (%)'].mean()
                avg_mdd_combined = combined_df['MDD (%)'].mean()
                avg_selected_stock_combined = combined_df['入選股數平均'].mean()
                condition_label = f"{'包含' if include_condition else '不包含'}{condition_name} (綜合)"
                summary_data.append([condition_label, avg_cagr_combined, avg_mdd_combined, avg_selected_stock_combined])

    # 返回新的 DataFrame
    summary_df = pd.DataFrame(summary_data, columns=['Group', 'Average CAGR (%)', 'Average MDD (%)', 'Average Selected Stock Count'])
    return summary_df

# signal_dict_comb = sim_conditions(signal_dict, resample='ME', data=data)
summary_df = analyze_signal_groups(signal_dict_comb.selected_stock_count_analysis())
summary_df

### 繪圖

In [ ]:
# 創建DataFrame
df = summary_df.copy()

# # 處理數據：提取所需的行（包含/不包含且是含本益比進出場的組合）
# mask = df['Group'].str.contains('含本益比進出場')
# filtered_df = df[mask].copy()

# # 定義類別順序
# categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# # 創建類別對應的數據
# included_values = []
# excluded_values = []

# for category in categories:
#     # 獲取包含該類別的資料
#     included_mask = filtered_df['Group'].str.contains(f'包含{category}')
#     excluded_mask = filtered_df['Group'].str.contains(f'不包含{category}')
    
#     included_value = filtered_df[included_mask]['Average CAGR (%)'].values[0]
#     excluded_value = filtered_df[excluded_mask]['Average CAGR (%)'].values[0]
    
#     included_values.append(included_value)
#     excluded_values.append(excluded_value)

# # 設置圖表風格和大小
# # plt.style.use('seaborn')
# plt.figure(figsize=(14, 6))

# # 設置bar的寬度
# width = 0.17

# # 設置x軸位置
# x = np.arange(len(categories))

# # 繪製條形圖
# bars1 = plt.bar(x - width/2, included_values, width, label='包含', color='tab:blue')
# bars2 = plt.bar(x + width/2, excluded_values, width, label='不包含', color='orange')

# # 設置圖表標題和標籤
# plt.title('台股 有無選股原則中其中一項因子的組合 CAGR 平均 (含本益比進出場)', fontsize=12)
# plt.xlabel('指標類別')
# plt.ylabel('Average CAGR (%)')

# # 設置x軸刻度和標籤
# plt.xticks(x, categories)

# # 設置y軸範圍
# plt.ylim(0, 16)

# # 添加網格線
# plt.grid(True, linestyle='--', alpha=0.8)

# benchmark_value = signal_dict_comb.selected_stock_count_analysis().loc['ROE五年平均+盈再率+三年配息率+稅後淨利_董監持股+上市櫃滿兩年_本益比進出場']['CAGR (%)']
# plt.axhline(y=benchmark_value, color='red', linestyle='--', label='所有條件+本益比進出場 CAGR', linewidth=0.3)

# # 添加圖例
# plt.legend(loc='upper center', bbox_to_anchor=(1.15, 1), fontsize=10)

# # 調整布局
# plt.tight_layout()

# # 顯示圖表
# plt.show()

# 定義類別順序
categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# 創建類別對應的數據
included_no_pe_values = []  # 包含但不含本益比
included_with_pe_values = []  # 包含且含本益比
excluded_no_pe_values = []  # 不包含且不含本益比
excluded_with_pe_values = []  # 不包含且含本益比

for category in categories:
    # 包含且不含本益比
    mask_included_no_pe = df['Group'].str.contains(f'包含{category} \(不含本益比進出場\)')
    included_no_pe_values.append(df[mask_included_no_pe]['Average CAGR (%)'].values[0])
    
    # 包含且含本益比
    mask_included_with_pe = df['Group'].str.contains(f'包含{category} \(含本益比進出場\)')
    included_with_pe_values.append(df[mask_included_with_pe]['Average CAGR (%)'].values[0])
    
    # 不包含且不含本益比
    mask_excluded_no_pe = df['Group'].str.contains(f'不包含{category} \(不含本益比進出場\)')
    excluded_no_pe_values.append(df[mask_excluded_no_pe]['Average CAGR (%)'].values[0])
    
    # 不包含且含本益比
    mask_excluded_with_pe = df['Group'].str.contains(f'不包含{category} \(含本益比進出場\)')
    excluded_with_pe_values.append(df[mask_excluded_with_pe]['Average CAGR (%)'].values[0])

# 設置圖表大小
fig = plt.figure(figsize=(10, 6))

# 設置bar的寬度
width = 0.17

# 設置x軸位置
x = np.arange(len(categories))

# 繪製條形圖
bars1 = plt.bar(x - width*1.5, included_no_pe_values, width, 
                label='包含某項因子_無本益比進出場', color='lightblue')
bars2 = plt.bar(x - width/2, included_with_pe_values, width, 
                label='包含某項因子_加上本益比進出場', color='tab:blue')
bars3 = plt.bar(x + width/2, excluded_no_pe_values, width, 
                label='不包含某項因子_不含本益比', color='bisque')
bars4 = plt.bar(x + width*1.5, excluded_with_pe_values, width, 
                label='不包含某項因子_加上本益比進出場', color='orange')

# 設置圖表標題和標籤
plt.title('美股 有無選股原則中其中一項因子的組合 CAGR 平均 (2003-2024 resample="M")', fontsize=12)
plt.xlabel('指標類別')
plt.ylabel('平均 CAGR (%)', fontsize=12)

# 設置x軸刻度和標籤
plt.xticks(x, categories, fontsize=12)

# 設置y軸範圍
plt.ylim(0, 18)

# # 添加網格線
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', linestyle='--', alpha=0.8)

# 添加基準線
benchmark_value = signal_dict_comb.selected_stock_count_analysis().loc['ROE五年平均+盈再率+三年配息率+稅後淨利_上市櫃滿兩年_本益比進出場']['CAGR (%)']
plt.axhline(y=benchmark_value, color='red', linestyle='--', 
            label='所有條件+本益比進出場 CAGR', linewidth=0.4)

# # 在bar上添加數值標籤
# def add_value_labels(bars):
#     for bar in bars:
#         height = bar.get_height()
#         plt.text(bar.get_x() + bar.get_width()/2., height,
#                 f'{height:.2f}%',
#                 ha='center', va='bottom',
#                 fontsize=8)

# add_value_labels(bars1)
# add_value_labels(bars2)
# add_value_labels(bars3)
# add_value_labels(bars4)

# 添加圖例
plt.legend()

# 調整布局
plt.tight_layout()

# 顯示圖表
plt.show()

fig.savefig('./img/圖 41美股有無其中一項因子的策略組合之CAGR平均.svg', format='svg', bbox_inches='tight')

In [ ]:
# 創建DataFrame
df = summary_df.copy()


# 定義類別順序
categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# 創建類別對應的數據
included_no_pe_values = []  # 包含但不含本益比
included_with_pe_values = []  # 包含且含本益比
excluded_no_pe_values = []  # 不包含且不含本益比
excluded_with_pe_values = []  # 不包含且含本益比

for category in categories:
    # 包含且不含本益比
    mask_included_no_pe = df['Group'].str.contains(f'包含{category} \(不含本益比進出場\)')
    included_no_pe_values.append(df[mask_included_no_pe]['Average Selected Stock Count'].values[0])
    
    # 包含且含本益比
    mask_included_with_pe = df['Group'].str.contains(f'包含{category} \(含本益比進出場\)')
    included_with_pe_values.append(df[mask_included_with_pe]['Average Selected Stock Count'].values[0])
    
    # 不包含且不含本益比
    mask_excluded_no_pe = df['Group'].str.contains(f'不包含{category} \(不含本益比進出場\)')
    excluded_no_pe_values.append(df[mask_excluded_no_pe]['Average Selected Stock Count'].values[0])
    
    # 不包含且含本益比
    mask_excluded_with_pe = df['Group'].str.contains(f'不包含{category} \(含本益比進出場\)')
    excluded_with_pe_values.append(df[mask_excluded_with_pe]['Average Selected Stock Count'].values[0])

# 設置圖表大小
fig = plt.figure(figsize=(10, 6))

# 設置bar的寬度
width = 0.17

# 設置x軸位置
x = np.arange(len(categories))

# 繪製條形圖
bars1 = plt.bar(x - width*1.5, included_no_pe_values, width, 
                label='包含某項因子_無本益比進出場', color='lightblue')
bars2 = plt.bar(x - width/2, included_with_pe_values, width, 
                label='包含某項因子_加上本益比進出場', color='tab:blue')
bars3 = plt.bar(x + width/2, excluded_no_pe_values, width, 
                label='不包含某項因子_不含本益比', color='bisque')
bars4 = plt.bar(x + width*1.5, excluded_with_pe_values, width, 
                label='不包含某項因子_含本益比', color='orange')

# 設置圖表標題和標籤
plt.title('美股 有無選股原則中其中一項因子的組合_入選股數平均 (2003-2024 resample="M")', fontsize=12)
plt.xlabel('指標類別')
plt.ylabel('入選股數平均', fontsize=12)

# 設置x軸刻度和標籤
plt.xticks(x, categories, fontsize=12)

# # 設置y軸範圍
# plt.ylim(0, 16)

# # 添加網格線
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', linestyle='--', alpha=0.8)



# 添加圖例
plt.legend()

# 調整布局
plt.tight_layout()

# 顯示圖表
plt.show()

fig.savefig('./img/圖 42 美股有無其中一項因子的策略組合之平均入選股數.svg', format='svg', bbox_inches='tight')

In [ ]:
# # 創建DataFrame
# df = summary_df

# # 設定中文字型
# plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
# plt.rcParams['axes.unicode_minus'] = False

# # 處理數據：提取所需的行（包含/不包含且是含本益比進出場的組合）
# mask = df['Group'].str.contains('不含本益比進出場')
# filtered_df = df[mask].copy()

# # 定義類別順序
# categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# # 創建類別對應的數據
# included_values = []
# excluded_values = []

# for category in categories:
#     # 獲取包含該類別的資料
#     included_mask = filtered_df['Group'].str.contains(f'包含{category}')
#     excluded_mask = filtered_df['Group'].str.contains(f'不包含{category}')
    
#     included_value = filtered_df[included_mask]['Average CAGR (%)'].values[0]
#     excluded_value = filtered_df[excluded_mask]['Average CAGR (%)'].values[0]
    
#     included_values.append(included_value)
#     excluded_values.append(excluded_value)

# # 設置圖表風格和大小
# # plt.style.use('seaborn')
# plt.figure(figsize=(14, 5))

# # 設置bar的寬度
# width = 0.17

# # 設置x軸位置
# x = np.arange(len(categories))

# # 繪製條形圖
# bars1 = plt.bar(x - width/2, included_values, width, label='包含', color='tab:blue')
# bars2 = plt.bar(x + width/2, excluded_values, width, label='不包含', color='orange')

# # 設置圖表標題和標籤
# plt.title('台股 有無選股原則中其中一項因子的組合 CAGR 平均 (不含本益比進出場)', fontsize=12)
# plt.xlabel('指標類別')
# plt.ylabel('Average CAGR (%)')

# # 設置x軸刻度和標籤
# plt.xticks(x, categories)



# # 設置y軸範圍
# plt.ylim(0, 16)

# # 添加網格線
# plt.grid(True, linestyle='--', alpha=0.8)

# benchmark_value=signal_dict_comb.selected_stock_count_analysis().loc['ROE五年平均+盈再率+三年配息率+稅後淨利_董監持股+上市櫃滿兩年']['CAGR (%)']
# plt.axhline(y=benchmark_value, color='red', linestyle='--', label='所有條件_沒有本益比進出場CAGR', linewidth=0.3)

# # 添加圖例
# plt.legend(loc='upper center', bbox_to_anchor=(1.15, 1), fontsize=12)


# # 調整布局
# plt.tight_layout()

# # 顯示圖表
# plt.show()

---

# TODO

## 選股策略使用不同篩選標準

#### memo

In [ ]:
# # 無本益比進出場 #
# # 稅後淨利條件
# orig_all_cond = (roe_cond & rr_cond & payout_cond & netprofit_cond & listed_cond)[START_DATE:END_DATE]
# # 稅前淨利條件
# orig_all_cond_inc_bf_tax = (roe_cond & rr_cond & payout_cond & income_bf_tax_cond & listed_cond)[START_DATE:END_DATE]

# # 有本益比進出場 #
# # 每季本益比 #
# # 稅後淨利條件
# orig_all_cond_and_pe = ((orig_all_cond & pe_cond_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_cond_exit[START_DATE:END_DATE]))
# # 稅前淨利條件
# orig_all_cond_and_pe_inc_bf_tax = ((orig_all_cond_inc_bf_tax & pe_cond_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond_inc_bf_tax) | pe_cond_exit[START_DATE:END_DATE]))

# # 每月月底本益比 #
# # 稅後淨利條件
# orig_all_cond_and_pe_daily = ((orig_all_cond & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE]))
# # 稅前淨利條件
# orig_all_cond_and_pe_daily_inc_bf_tax = ((orig_all_cond_inc_bf_tax & daily_pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond_inc_bf_tax) | daily_pe_exit[START_DATE:END_DATE]))

---

In [ ]:
def plot_strategy_comparison(df, compare='CAGR (%)', title=None, order_list=None, pattern=False):
    """
    繪製策略比較的柱狀圖
    
    Parameters:
    -----------
    df : pandas DataFrame
        包含策略資訊的數據框
    compare : str, default='CAGR (%)'
        要比較的指標欄位名稱
    title : str, default=None
        自訂圖表標題，若為None則使用預設標題
    order_list : list, default=None
        指定策略前綴的顯示順序，若為None則使用原始順序
    pattern : bool, default=False
        是否顯示子標籤模式
    """
    import matplotlib.pyplot as plt
    import re
    
    # 取得所有策略名稱並分組
    strategies = df['Strategy'].tolist()
    strategy_pairs = {}
    
    for strategy in strategies:
        if '有本益比進出場' in strategy:
            prefix = strategy.replace('有本益比進出場', '')
            strategy_type = '有本益比'
        else:
            prefix = strategy.replace('無本益比進出場', '')
            strategy_type = '無本益比'
            
        if prefix not in strategy_pairs:
            strategy_pairs[prefix] = {}
        strategy_pairs[prefix][strategy_type] = df[df['Strategy'] == strategy][compare].values[0]

    # 如果有指定順序，重新排序strategy_pairs
    if order_list is not None:
        ordered_pairs = {k: strategy_pairs[k] for k in order_list if k in strategy_pairs}
        strategy_pairs = ordered_pairs

    # 設定圖表
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 設定柱狀圖位置
    x = range(len(strategy_pairs))
    width = 0.35
    
    # 繪製柱狀圖
    rects1 = ax.bar([i - width/2 for i in x], 
                    [pair['有本益比'] for pair in strategy_pairs.values()],
                    width, label='有本益比')
    
    rects2 = ax.bar([i + width/2 for i in x], 
                    [pair['無本益比'] for pair in strategy_pairs.values()],
                    width, label='無本益比')

    # 設定主要x軸標籤
    ax.set_ylabel(compare)
    ax.set_title(title if title else f'策略比較 - {compare}')
    ax.set_xticks(x)
    ax.set_xticklabels(strategy_pairs.keys(), rotation=45, ha='right', fontsize=14)

    # 處理pattern模式的子標籤
    if pattern:
        # 創建第二個x軸
        ax2 = ax.twiny()
        ax2.spines['top'].set_position(('axes', 1.0))
        
        # 找出所有前綴模式
        prefixes = list(strategy_pairs.keys())
        patterns = set()
        for prefix in prefixes:
            match = re.match(r'([^_]+_[^_]+)_.*', prefix)
            if match:
                patterns.add(match.group(1))
        
        patterns = sorted(list(patterns))
        pattern_positions = {}
        
        # 計算每個模式的平均位置
        for pattern in patterns:
            pattern_indices = [i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]
            if pattern_indices:
                pattern_positions[pattern] = sum(pattern_indices) / len(pattern_indices)
        
        # 設定子標籤
        ax2.set_xlim(ax.get_xlim())
        ax2.set_xticks([pattern_positions[pattern] for pattern in patterns])
        ax2.set_xticklabels(patterns, rotation=0, ha='center', fontsize=14)
        
        # 添加垂直分隔線
        ax.grid(False)  # 關閉默認格線
        
        # 獲取y軸的範圍
        ymin, ymax = ax.get_ylim()
        
        # 在每個pattern的起始和結束位置添加垂直線
        for pattern in patterns:
            pattern_start = min([i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]) - 0.5
            pattern_end = max([i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]) + 0.5
            
            # # 添加淺色背景區塊
            # ax.axvspan(pattern_start, pattern_end, 
            #           color='gray', alpha=0.1)
            
            # 添加垂直分隔線
            ax.axvline(x=pattern_start, color='gray', 
                      linestyle='--', alpha=0.7, linewidth=0.5)
            ax.axvline(x=pattern_end, color='gray', 
                      linestyle='--', alpha=0.7, linewidth=0.5)
        
    ax.legend(loc='upper right')
    plt.tight_layout()
    
    plt.show()

    fig.savefig(f'./img/{title}.svg', format='svg', bbox_inches='tight')

### 其他條件固定，ROE參數最佳化

In [ ]:
roe_value_cond = {}
no_roe_conds = (rr_cond & netprofit_cond & payout_cond & listed_cond)[START_DATE:END_DATE]

for i in range(10, 31, 5): # 大於 10~30%
    for n in range(3, 6): # 3, 4, 5年平均
        

        roe_opt_df = roe.copy()
        roe_opt_df['month'] = roe_opt_df.index.month

        roe_df_result = roe_opt_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(n, min_periods=n).mean()) #, include_groups=False)

        roe_cond_opt = (roe_df_result > (i/100))[START_DATE:END_DATE]

        roe_value_cond[f'roe_{n}y_{i}_無本益比進出場'] = (roe_cond_opt & no_roe_conds)[START_DATE:END_DATE]
        roe_value_cond[f'roe_{n}y_{i}_有本益比進出場'] = ((roe_cond_opt & no_roe_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(roe_cond_opt & no_roe_conds)) | daily_pe_exit[START_DATE:END_DATE]))
        
roe_collection = sim_conditions(roe_value_cond, resample='M', data=data)
roe_collection.selected_stock_count_analysis()

In [ ]:
roe_collection_df = roe_collection.selected_stock_count_analysis()
roe_collection_df.reset_index(inplace=True)

In [ ]:
# 建立顏色映射
colors = {
    '10': 'tab:blue',
    '15': 'orange',
    '20': 'tab:green',
    '25': 'tab:red', 
    '30': 'tab:purple'
}

# 提取年份和ROE閾值
roe_collection_df['Year'] = roe_collection_df['Strategy'].str.extract(r'roe_(\d)y')
roe_collection_df['ROE'] = roe_collection_df['Strategy'].str.extract(r'_(\d+)_')
roe_collection_df['PE'] = roe_collection_df['Strategy'].str.contains('有本益比')

In [ ]:
# 獲取唯一的年份值
years = sorted(roe_collection_df['Year'].unique())

# 設定長條的寬度
bar_width = 0.15

# 計算每個年份組的位置
positions = np.arange(len(years))

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

fig, ax1 = plt.subplots(figsize=(14, 10))

# 儲存所有CAGR值用於設定y軸範圍
roeopt_cagr_values = []

# 儲存有本益比和無本益比的CAGR值
cagr_with_pe = []
cagr_without_pe = []

# 繪製有本益比的長條圖
bars = []  # 儲存長條物件用於之後設定legend
for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
    mask_with_pe = (roe_collection_df['ROE'] == roe_bound) & (roe_collection_df['PE'])
    data_with_pe = [roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
                    if len(roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
                    for year in years]
    roeopt_cagr_values.extend(data_with_pe)
    cagr_with_pe.extend(data_with_pe)
    
    # 計算長條的位置
    x = positions + (i - 1.5) * bar_width

    # 繪製長條
    bar = ax1.bar(x, data_with_pe, bar_width, label=f'ROE {roe_bound}%', color=colors[roe_bound])
    bars.append(bar)

# # 繪製無本益比的長條圖
# for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
#     mask_without_pe = (roe_collection_df['ROE'] == roe_bound) & (~roe_collection_df['PE'])
#     data_without_pe = [roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
#                       if len(roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
#                       for year in years]
#     roeopt_cagr_values.extend(data_without_pe)
#     cagr_without_pe.extend(data_without_pe)
    
#     # 計算長條的位置
#     x = positions + (i - 1.5) * bar_width

#     # 繪製長條
#     ax2.bar(x, data_without_pe, bar_width, color=colors[roe_bound])

# 計算CAGR平均數
avg_cagr_with_pe = sum(cagr_with_pe) / len(cagr_with_pe)
# avg_cagr_without_pe = sum(cagr_without_pe) / len(cagr_without_pe)

# 設定兩個子圖的共同y軸範圍
# ax1.set_ylim(0, 14.5)
# ax2.set_ylim(0, 14.5)

# 設定左方子圖（有本益比）
ax1.set_xticks(positions)
ax1.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=18)
ax1.set_title(f"美股_ROE變化_有本益比進出場策略的CAGR比較", fontsize=20)
ax1.set_xlabel('平均年份', fontsize=18, labelpad=10)
ax1.set_ylabel('CAGR (%)', fontsize=16)
ax1.grid(axis='y', linestyle='--', alpha=0.6)

# # 設定右方子圖（無本益比）
# ax2.set_xticks(positions)
# ax2.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=16)
# ax2.set_title(f"美股_ROE變化_無本益比進出場策略的CAGR比較", fontsize=16) # \nCAGR平均: {avg_cagr_without_pe:.2f}%'
# ax2.set_ylabel('CAGR (%)', fontsize=12)
# ax2.grid(axis='y', linestyle='--', alpha=0.6)

# 調整版面配置
plt.tight_layout()

# 將legend放在整個圖表的最左側
legend = fig.legend(bars, 
                   [f'ROE N 年平均 > {roe_bound}%' for roe_bound in ['10', '15', '20', '25', '30']], 
                   loc='upper left',
                   bbox_to_anchor=(-0.2, 0.95),  # 調整這些數值以微調legend位置
                   fontsize=16,
                   title="平均ROE標準", title_fontsize=18)


# # 在兩圖中間下方添加legend
# legend = fig.legend(bars, [f'ROE N 年平均 > {roe_bound}%' for roe_bound in ['10', '15', '20', '25', '30']], 
#                    loc='center', bbox_to_anchor=(0.5, 0.02),
#                    ncol=5, frameon=False, fontsize=14)

# 調整子圖之間的間距和底部空間
plt.subplots_adjust(bottom=0.1)  # 為legend留出空間

# 顯示圖表
plt.show()

fig.savefig('./img/圖 52美股有進出場條件策略ROE變化CAGR比較圖.svg', dpi=300, bbox_inches='tight')

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# roeopt_mdd_values = []

# # 儲存有本益比和無本益比的MDD值
# mdd_with_pe = []
# mdd_without_pe = []

# # 繪製有本益比的長條圖
# bars = []  # 儲存長條物件用於之後設定legend
# for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
#     mask_with_pe = (roe_collection_df['ROE'] == roe_bound) & (roe_collection_df['PE'])
#     data_with_pe = [roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]['MDD (%)'].values[0] 
#                     if len(roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
#                     for year in years]
#     roeopt_mdd_values.extend(data_with_pe)
#     mdd_with_pe.extend(data_with_pe)
    
#     # 計算長條的位置
#     x = positions + (i - 1.5) * bar_width

#     # 繪製長條
#     bar = ax1.bar(x, data_with_pe, bar_width, label=f'ROE {roe_bound}%', color=colors[roe_bound])
#     bars.append(bar)

# # 繪製無本益比的長條圖
# for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
#     mask_without_pe = (roe_collection_df['ROE'] == roe_bound) & (~roe_collection_df['PE'])
#     data_without_pe = [roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]['MDD (%)'].values[0] 
#                       if len(roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
#                       for year in years]
#     roeopt_mdd_values.extend(data_without_pe)
#     mdd_without_pe.extend(data_without_pe)
    
#     # 計算長條的位置
#     x = positions + (i - 1.5) * bar_width

#     # 繪製長條
#     ax2.bar(x, data_without_pe, bar_width, color=colors[roe_bound])

# # 計算MDD平均數
# avg_mdd_with_pe = sum(mdd_with_pe) / len(mdd_with_pe)
# avg_mdd_without_pe = sum(mdd_without_pe) / len(mdd_without_pe)

# # 設定兩個子圖的共同y軸範圍
# ax1.set_ylim(-75, 0)
# ax2.set_ylim(-75, 0)

# # 設定左方子圖（有本益比）
# ax1.set_xticks(positions)
# ax1.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=14)
# ax1.set_title(f'美股_ROE變化_有本益比進出場策略的MDD比較\nMDD平均: {avg_mdd_with_pe:.2f}%', fontsize=14, pad=20)
# ax1.set_ylabel('MDD (%)', fontsize=12)
# ax1.grid(axis='y', linestyle='--', alpha=0.6)

# # 設定右方子圖（無本益比）
# ax2.set_xticks(positions)
# ax2.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=14)
# ax2.set_title(f'美股_ROE變化_無本益比進出場策略的MDD比較\nMDD平均: {avg_mdd_without_pe:.2f}%', fontsize=14, pad=20)
# ax2.set_ylabel('MDD (%)', fontsize=12)
# ax2.grid(axis='y', linestyle='--', alpha=0.6)

# # 調整版面配置
# plt.tight_layout()

# # 在兩圖中間下方添加legend
# legend = fig.legend(bars, [f'ROE N 年平均 > {roe_bound}%' for roe_bound in ['10', '15', '20', '25', '30']], 
#                    loc='center', bbox_to_anchor=(0.5, 0.02),
#                    ncol=5, frameon=False, fontsize=14)

# # 調整子圖之間的間距和底部空間
# plt.subplots_adjust(bottom=0.1)  # 為legend留出空間

# # 顯示圖表
# plt.show()

#### 入選股數平均

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# # 儲存所有CAGR值用於設定y軸範圍
# roeopt_stock_values = []

# # 繪製有本益比的長條圖
# bars = []  # 儲存長條物件用於之後設定legend
# for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
#     mask_with_pe = (roe_collection_df['ROE'] == roe_bound) & (roe_collection_df['PE'])
#     data_with_pe = [roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]['入選股數平均'].values[0] 
#                     if len(roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
#                     for year in years]
#     roeopt_stock_values.extend(data_with_pe)
    
#     # 計算長條的位置
#     x = positions + (i - 1.5) * bar_width

#     # 繪製長條
#     bar = ax1.bar(x, data_with_pe, bar_width, label=f'ROE {roe_bound}%', color=colors[roe_bound])
#     bars.append(bar)

# # 繪製無本益比的長條圖
# for i, roe_bound in enumerate(['10', '15', '20', '25', '30']):
#     mask_without_pe = (roe_collection_df['ROE'] == roe_bound) & (~roe_collection_df['PE'])
#     data_without_pe = [roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]['入選股數平均'].values[0] 
#                       if len(roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
#                       for year in years]
#     roeopt_stock_values.extend(data_without_pe)
    
#     # 計算長條的位置
#     x = positions + (i - 1.5) * bar_width

#     # 繪製長條
#     ax2.bar(x, data_without_pe, bar_width, color=colors[roe_bound])

# # 設定兩個子圖的共同y軸範圍
# ax1.set_ylim(0,60)
# ax2.set_ylim(0,60)

# # 設定左方子圖（有本益比）
# ax1.set_xticks(positions)
# ax1.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=14)
# ax1.set_title('美股_ROE變化_有本益比進出場策略_入選股數平均', fontsize=14, pad=20)
# # ax1.set_xlabel('年份', fontsize=12)
# ax1.set_ylabel('數量', fontsize=12)
# ax1.grid(axis='y', linestyle='--', alpha=0.6)

# # 設定右方子圖（無本益比）
# ax2.set_xticks(positions)
# ax2.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=14)
# ax2.set_title('美股_ROE變化_無本益比進出場策略_入選股數平均', fontsize=14, pad=20)
# # ax2.set_xlabel('年份', fontsize=12)
# ax2.set_ylabel('數量', fontsize=12)
# ax2.grid(axis='y', linestyle='--', alpha=0.6)

# # 調整版面配置
# plt.tight_layout()

# # 在兩圖中間下方添加legend
# legend = fig.legend(bars, [f'ROE N 年平均 > {roe_bound}%' for roe_bound in ['10', '15', '20', '25', '30']], 
#                    loc='center', bbox_to_anchor=(0.5, 0.02),
#                    ncol=5, frameon=False, fontsize=14)

# # 調整子圖之間的間距和底部空間
# plt.subplots_adjust(bottom=0.1)  # 為legend留出空間

# # 顯示圖表
# plt.show()

In [ ]:
plot_strategy_comparison(roe_collection_df, 
                         title='美股_ROE變化_有無本益比進出場策略比較_CAGR(%)',
                         order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
                                     'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
                                     'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'],
                         pattern=True)

In [ ]:
plot_strategy_comparison(roe_collection_df,
                            compare='MDD (%)',
                            title='美股_ROE變化_有無本益比進出場策略比較_MDD(%)',
                            order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
                                     'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
                                     'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'],
                            pattern=True)

In [ ]:
# plot_strategy_comparison(roe_collection_df,
#                             compare='入選股數平均',
#                             title='美股_ROE變化_有無本益比進出場策略比較_入選股數平均',
#                             order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
#                                      'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
#                                      'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'])

### 其他條件固定，盈餘再投資率參數最佳化

In [ ]:
rr_value_cond = {}
no_rr_conds = (roe_cond & netprofit_cond & payout_cond & listed_cond)[START_DATE:END_DATE]

for r in range(0, 81, 10): # 小於 0~40% (80%)

    rr_opt_df = rr.copy()
    rr_cond_opt = (rr_opt_df < (r/100))[START_DATE:END_DATE]

    rr_value_cond[f'盈再率小於_{r}_無本益比進出場'] = (rr_cond_opt & no_rr_conds)[START_DATE:END_DATE]
    rr_value_cond[f'盈再率小於_{r}_有本益比進出場'] = ((rr_cond_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_cond_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

rr_collection = sim_conditions(rr_value_cond, resample='M', data=data)
rr_collection.selected_stock_count_analysis()

In [ ]:
rr_collection_df = rr_collection.selected_stock_count_analysis()
rr_collection_df.reset_index(inplace=True)

In [ ]:
rr_collection_df

改成柱狀圖

In [ ]:
# # 分離有無本益比的策略
# no_pe_mask = rr_collection_df['Strategy'].str.contains('無本益比')
# with_pe_mask = rr_collection_df['Strategy'].str.contains('有本益比')

# # 從策略名稱中提取百分比數值
# rr_collection_df['threshold'] = rr_collection_df['Strategy'].str.extract(r'小於_(\d+)').astype(float)

# # 排序數據
# no_pe_data = rr_collection_df[no_pe_mask].sort_values('threshold')
# with_pe_data = rr_collection_df[with_pe_mask].sort_values('threshold')

# # 繪圖設定
# plt.figure(figsize=(10, 6))

# # 繪製無本益比策略的折線
# plt.plot(no_pe_data['threshold'], 
#          no_pe_data['CAGR (%)'], 
#          marker='o', 
#          linestyle='-', 
#          label='無本益比進出場', 
#          linewidth=2)

# # 繪製有本益比策略的折線
# plt.plot(with_pe_data['threshold'], 
#          with_pe_data['CAGR (%)'], 
#          marker='s', 
#          linestyle='--', 
#          label='有本益比進出場', 
#          linewidth=2)

# # 設定圖表格式
# plt.title('美股_盈再率變化_CAGR(%)比較', fontsize=14)
# plt.xlabel('盈再率 (%)', fontsize=12)
# plt.ylabel('CAGR (%)', fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.7)
# plt.legend(fontsize=10)

# # 設定X軸刻度
# plt.xticks(no_pe_data['threshold'])

# # 顯示圖表
# plt.tight_layout()
# plt.show()

In [ ]:
plot_strategy_comparison(rr_collection_df,
                            compare='CAGR (%)',
                            title='美股_盈再率變化_有無本益比進出場策略比較_CAGR(%)',
                            order_list=['盈再率小於_0_', '盈再率小於_10_', '盈再率小於_20_', '盈再率小於_30_', '盈再率小於_40_', '盈再率小於_50_', '盈再率小於_60_', '盈再率小於_70_', '盈再率小於_80_'])

In [ ]:
plot_strategy_comparison(rr_collection_df,
                            compare='MDD (%)',
                            title='美股_盈再率變化_有無本益比進出場策略比較_MDD(%)',
                            order_list=['盈再率小於_0_', '盈再率小於_10_', '盈再率小於_20_', '盈再率小於_30_', '盈再率小於_40_', '盈再率小於_50_', '盈再率小於_60_', '盈再率小於_70_', '盈再率小於_80_'])

In [ ]:
# plot_strategy_comparison(rr_collection_df,
#                             compare='入選股數平均',
#                             title='美股_盈再率變化_有無本益比進出場策略比較_入選股數平均',
#                             order_list=['盈再率小於_0_', '盈再率小於_10_', '盈再率小於_20_', '盈再率小於_30_', '盈再率小於_40_', '盈再率小於_50_', '盈再率小於_60_', '盈再率小於_70_', '盈再率小於_80_'])

In [ ]:
# rr_test_cond = {}
# no_rr_conds = (roe_cond & netprofit_cond & payout_cond & listed_cond)[START_DATE:END_DATE]

# # 小於0%的情境
# rr_test_df = rr.copy()
# rr_test_opt = (rr_test_df < 0)[START_DATE:END_DATE]
# rr_test_cond['盈再率小於_0_無本益比進出場'] = (rr_test_opt & no_rr_conds)[START_DATE:END_DATE]
# rr_test_cond['盈再率小於_0_有本益比進出場'] = ((rr_test_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_test_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

# # 0%到80%，每20%一個區間
# for r in range(0, 61, 20):
#     rr_test_df = rr.copy()
#     rr_test_opt = ((rr_test_df >= (r/100)) & (rr_test_df < ((r+20)/100)))[START_DATE:END_DATE]
    
#     rr_test_cond[f'盈再率大於_{r}_小於_{r+20}_無本益比進出場'] = (rr_test_opt & no_rr_conds)[START_DATE:END_DATE]
#     rr_test_cond[f'盈再率大於_{r}_小於_{r+20}_有本益比進出場'] = ((rr_test_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_test_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

# # 大於80%的情境
# rr_test_opt = (rr_test_df >= 0.8)[START_DATE:END_DATE]
# rr_test_cond['盈再率大於_80_無本益比進出場'] = (rr_test_opt & no_rr_conds)[START_DATE:END_DATE]
# rr_test_cond['盈再率大於_80_有本益比進出場'] = ((rr_test_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_test_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

# rr_test_collection = sim_conditions(rr_test_cond, resample='ME', data=data)
# rr_test_collection.selected_stock_count_analysis()

In [ ]:
rr_test_cond = {}
no_rr_conds = (roe_cond & netprofit_cond & payout_cond & listed_cond)[START_DATE:END_DATE]

# 小於0%的情境
rr_test_df = rr.copy()
rr_test_opt = (rr_test_df < 0)[START_DATE:END_DATE]
rr_test_cond['盈再率小於_0'] = (rr_test_opt & no_rr_conds)[START_DATE:END_DATE]
# rr_test_cond['盈再率小於_0_有本益比進出場'] = ((rr_test_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_test_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

# 0%到80%，每20%一個區間
for r in range(0, 61, 20):
    rr_test_df = rr.copy()
    rr_test_opt = ((rr_test_df >= (r/100)) & (rr_test_df < ((r+20)/100)))[START_DATE:END_DATE]
    
    rr_test_cond[f'盈再率大於_{r}_小於_{r+20}'] = (rr_test_opt & no_rr_conds)[START_DATE:END_DATE]
    # rr_test_cond[f'盈再率大於_{r}_小於_{r+20}_有本益比進出場'] = ((rr_test_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_test_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

# 大於80%的情境
rr_test_opt = (rr_test_df >= 0.8)[START_DATE:END_DATE]
rr_test_cond['盈再率大於_80'] = (rr_test_opt & no_rr_conds)[START_DATE:END_DATE]
# rr_test_cond['盈再率大於_80_有本益比進出場'] = ((rr_test_opt & no_rr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(rr_test_opt & no_rr_conds)) | daily_pe_exit[START_DATE:END_DATE]))

rr_test_collection = sim_conditions(rr_test_cond, resample='M', data=data)

In [ ]:
rr_test_collection_df = rr_test_collection.selected_stock_count_analysis()
rr_test_collection_df.drop(columns=['CAGR (%)', 'MDD (%)', '25%', '75%'], inplace=True)

In [ ]:
rr_test_collection_df.sort_values('入選股數平均', ascending=False)

### 其他條件固定，配息率參數最佳化

In [ ]:
pr_value_cond = {}
no_pr_conds = (roe_cond & rr_cond & netprofit_cond & listed_cond)[START_DATE:END_DATE]

for k in range(0, 56, 5): # 大於 0~55%

    pr_cond_opt = (payout_ratio_rol.copy() > (k/100))[START_DATE:END_DATE]

    pr_value_cond[f'配息_三年至少{k}_無本益比進出場'] = (pr_cond_opt & no_pr_conds)[START_DATE:END_DATE]
    pr_value_cond[f'配息_三年至少{k}_有本益比進出場'] = ((pr_cond_opt & no_pr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(pr_cond_opt & no_pr_conds)) | daily_pe_exit[START_DATE:END_DATE]))
    
pr_collection = sim_conditions(pr_value_cond, resample='M', data=data)
pr_collection.selected_stock_count_analysis()

In [ ]:
# pr_value_cond = {}
# no_pr_conds = (roe_cond & rr_cond & netprofit_cond & listed_cond)[START_DATE:END_DATE]

# for year in range(1, 4): # 當年至少大於N%、兩年內、三年內
#     for k in range(0, 56, 5): # 大於 0~40%

#         pr_opt_df = payout_ratio.copy()
#         pr_opt_df['month'] = pr_opt_df.index.month

#         pr_df_result = pr_opt_df.groupby('month', group_keys=False).apply(lambda group: group.rolling(year, min_periods=year).min(), include_groups=False)

#         pr_cond_opt = (pr_df_result > (k/100))[START_DATE:END_DATE]

#         pr_value_cond[f'配息_{year}年_至少{k}_無本益比進出場'] = (pr_cond_opt & no_pr_conds)[START_DATE:END_DATE]
#         pr_value_cond[f'配息_{year}年_至少{k}_有本益比進出場'] = ((pr_cond_opt & no_pr_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~(pr_cond_opt & no_pr_conds)) | daily_pe_exit[START_DATE:END_DATE]))
        
# pr_collection = sim_conditions(pr_value_cond, resample='ME', data=data)
# pr_collection.selected_stock_count_analysis()

In [ ]:
pr_collection_df = pr_collection.selected_stock_count_analysis()
pr_collection_df.reset_index(inplace=True)

In [ ]:
plot_strategy_comparison(pr_collection_df,
                            title='美股_配息_有無本益比進出場策略比較_CAGR(%)',
                            order_list=['配息_三年至少0_', '配息_三年至少5_', '配息_三年至少10_', '配息_三年至少15_', '配息_三年至少20_', '配息_三年至少25_', '配息_三年至少30_', '配息_三年至少35_', '配息_三年至少40_', '配息_三年至少45_', '配息_三年至少50_', '配息_三年至少55_'],
                            )

In [ ]:
plot_strategy_comparison(pr_collection_df,
                            compare='MDD (%)',
                            title='美股_配息_有無本益比進出場策略比較_MDD(%)',
                            order_list=['配息_三年至少0_', '配息_三年至少5_', '配息_三年至少10_', '配息_三年至少15_', '配息_三年至少20_', '配息_三年至少25_', '配息_三年至少30_', '配息_三年至少35_', '配息_三年至少40_', '配息_三年至少45_', '配息_三年至少50_', '配息_三年至少55_'],
                            )

In [ ]:
# plot_strategy_comparison(pr_collection_df,
#                             compare='入選股數平均',
#                             title='美股_配息率變化_有無本益比進出場策略比較_入選股數平均',
#                             order_list=['配息_三年至少0_', '配息_三年至少5_', '配息_三年至少10_', '配息_三年至少15_', '配息_三年至少20_', '配息_三年至少25_', '配息_三年至少30_', '配息_三年至少35_', '配息_三年至少40_', '配息_三年至少45_', '配息_三年至少50_', '配息_三年至少55_'])

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# # 資料處理
# # 提取年數和門檻值
# pr_collection_df['years'] = pr_collection_df['Strategy'].str.extract(r'配息_(\d)年').astype(int)
# pr_collection_df['threshold'] = pr_collection_df['Strategy'].str.extract(r'至少(\d+)').astype(int)

# # 分離有無本益比的資料，並過濾掉Min為0的資料
# no_pe_data = pr_collection_df[pr_collection_df['Strategy'].str.contains('無本益比')]
# with_pe_data = pr_collection_df[pr_collection_df['Strategy'].str.contains('有本益比')]
# no_pe_data = no_pe_data[no_pe_data['Min'] != 0]
# with_pe_data = with_pe_data[with_pe_data['Min'] != 0]

# # 獲取排序後的門檻值和年份
# thresholds = sorted(pr_collection_df['threshold'].unique())
# years = sorted(pr_collection_df['years'].unique())

# # 設定線型和顏色
# line_styles = ['-', '--', '-.']
# colors = ['tab:blue', 'tab:orange', 'green']

# # 繪製有本益比進出場的子圖 (左側)
# for idx, year in enumerate(years):
#     year_data = with_pe_data[with_pe_data['years'] == year].sort_values('threshold')
#     if not year_data.empty:  # 確保有資料才繪圖
#         ax1.plot(year_data['threshold'],
#                  year_data['CAGR (%)'],
#                  label=f'{year}年',
#                  linestyle=line_styles[idx],
#                  color=colors[idx],
#                  marker='o',
#                  markersize=3,
#                  linewidth=1)

# # 繪製無本益比進出場的子圖 (右側)
# for idx, year in enumerate(years):
#     year_data = no_pe_data[no_pe_data['years'] == year].sort_values('threshold')
#     if not year_data.empty:  # 確保有資料才繪圖
#         ax2.plot(year_data['threshold'], 
#                  year_data['CAGR (%)'],
#                  label=f'{year}年',
#                  linestyle=line_styles[idx],
#                  color=colors[idx],
#                  marker='o',
#                  markersize=3,
#                  linewidth=1)

# # 設定子圖1的格式 (左側 - 有本益比)
# ax1.set_title('有本益比進出場', fontsize=12)
# ax1.set_xlabel('配息率 (%)', fontsize=10)
# ax1.set_ylabel('CAGR (%)', fontsize=10)
# ax1.grid(True, linestyle='--', alpha=0.7)
# ax1.legend(title='年期')

# # 設定子圖2的格式 (右側 - 無本益比)
# ax2.set_title('無本益比進出場', fontsize=12)
# ax2.set_xlabel('配息率 (%)', fontsize=10)
# ax2.set_ylabel('CAGR (%)', fontsize=10)
# ax2.grid(True, linestyle='--', alpha=0.7)
# ax2.legend(title='年期')

# # 設定X軸刻度
# ax1.set_xticks(thresholds)
# ax2.set_xticks(thresholds)

# ax1.set_ylim(8, 12.5)
# ax2.set_ylim(8, 12.5)

# # 調整圖表整體佈局
# plt.suptitle('美股_配息率變化_CAGR(%)比較', fontsize=14)
# plt.tight_layout()
# plt.show()

In [ ]:
# pr_collection.reports['配息_1年_至少55_無本益比進出場'].display()
# pr_collection.reports['配息_3年_至少55_無本益比進出場'].trades

### 其他條件固定，PE進場條件參數最佳化

In [ ]:
pe_entry_value_cond = {}

# orig_all_cond
# daily_pe_entry = (pe_daily < 12).resample('ME').last()

for pe_entry_value in range(8, 13, 2): # 小於 8~12

    pe_entry_opt_df = (pe_daily < pe_entry_value).resample('M').last()[START_DATE:END_DATE]

    pe_entry_value_cond[f'pe小於_{pe_entry_value}進場'] = ((orig_all_cond & pe_entry_opt_df).hold_until((~orig_all_cond) | daily_pe_exit[START_DATE:END_DATE]))

pe_entry_collection = sim_conditions(pe_entry_value_cond, resample='M', data=data)
pe_entry_collection.selected_stock_count_analysis()

In [ ]:
pe_entry_collection.reports['pe小於_8進場'].display()

In [ ]:
# pe_entry_collection.reports['pe小於_8進場'].display()

---

## 兩兩一組

In [ ]:
# def plot_strategy_heatmap(df, compare='CAGR (%)', x_first=True, figsize=(14, 6), title=None):
#     """
#     繪製策略熱力圖
    
#     Parameters:
#     -----------
#     df : pandas DataFrame
#         包含 'Strategy' 和比較欄位的數據框
#     compare : str, default='CAGR (%)'
#         要比較的欄位名稱，例如 'CAGR (%)' 或 'MDD (%)'
#     x_first : bool, default=True
#         True: 條件1為X軸，條件2為Y軸
#         False: 條件1為Y軸，條件2為X軸
#     figsize : tuple, default=(14, 6)
#         圖形尺寸
#     title : str, optional
#         圖表標題，如果不指定則自動生成
#     """
    
#     # 檢查比較欄位是否存在
#     if compare not in df.columns:
#         raise ValueError(f"Column '{compare}' not found in DataFrame")
    
#     # 初始化列表存儲參數
#     param1_values = []
#     param2_values = []
    
#     # 從策略名稱中提取參數
#     for strategy in df['Strategy']:
#         parts = strategy.split('_')
#         param1 = float(parts[1].replace('%', ''))
#         param2 = float(parts[3].replace('%', ''))
#         param1_values.append(param1)
#         param2_values.append(param2)
    
#     # 將提取的參數加入 DataFrame
#     df['Param1'] = param1_values
#     df['Param2'] = param2_values
    
#     # 獲取條件名稱
#     condition1_name = df['Strategy'].iloc[0].split('_')[0]
#     condition2_name = df['Strategy'].iloc[0].split('_')[2]
    
#     # 創建樞紐表
#     if x_first:
#         pivot_table = df.pivot(
#             index='Param2',
#             columns='Param1',
#             values=compare
#         )
#         xlabel = f"{condition1_name} (%)"
#         ylabel = f"{condition2_name} (%)"
#     else:
#         pivot_table = df.pivot(
#             index='Param1',
#             columns='Param2',
#             values=compare
#         )
#         # 當x_first=False時，反轉index順序
#         pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
#         xlabel = f"{condition2_name} (%)"
#         ylabel = f"{condition1_name} (%)"
    
#     # 設置圖形
#     plt.figure(figsize=figsize)
    
#     # 繪製熱力圖
#     sns.heatmap(pivot_table,
#                 annot=True,
#                 annot_kws={'size': 14},
#                 fmt='.2f',
#                 cmap='coolwarm',
#                 cbar_kws={'label': compare},  # 使用比較欄位名稱作為色階標籤
#                 square=True
#     )
    
#     # 設置標題和軸標籤
#     if title is None:
#         title = f'{condition1_name} vs {condition2_name} 策略{compare}比較'
#     plt.title(title)
#     plt.xlabel(xlabel)
#     plt.ylabel(ylabel)
    
#     # 調整布局
#     plt.tight_layout()
#     plt.show()

In [ ]:
# def plot_strategy_heatmap(df, compare='CAGR (%)', x_first=True, figsize=(14, 6), title=None):
#     """
#     繪製策略熱力圖，當df['Min']==0時顯示灰色

#     Parameters:
#     -----------
#     df : pandas DataFrame
#         包含 'Strategy' 和比較欄位的數據框
#     compare : str, default='CAGR (%)'
#         要比較的欄位名稱，例如 'CAGR (%)' 或 'MDD (%)'
#     x_first : bool, default=True
#         True: 條件1為X軸，條件2為Y軸
#         False: 條件1為Y軸，條件2為X軸
#     figsize : tuple, default=(14, 6)
#         圖形尺寸
#     title : str, optional
#         圖表標題，如果不指定則自動生成
#     """
    
#     # 檢查比較欄位是否存在
#     if compare not in df.columns:
#         raise ValueError(f"Column '{compare}' not found in DataFrame")
    
#     # 初始化列表存儲參數
#     param1_values = []
#     param2_values = []
    
#     # 從策略名稱中提取參數
#     for strategy in df['Strategy']:
#         parts = strategy.split('_')
#         param1 = float(parts[1].replace('%', ''))
#         param2 = float(parts[3].replace('%', ''))
#         param1_values.append(param1)
#         param2_values.append(param2)
    
#     # 將提取的參數加入 DataFrame
#     df['Param1'] = param1_values
#     df['Param2'] = param2_values
    
#     # 獲取條件名稱
#     condition1_name = df['Strategy'].iloc[0].split('_')[0]
#     condition2_name = df['Strategy'].iloc[0].split('_')[2]

#     # 創建樞紐表 - 比較值
#     if x_first:
#         pivot_table = df.pivot(
#             index='Param2',
#             columns='Param1',
#             values=compare
#         )
#         min_pivot = df.pivot(
#             index='Param2',
#             columns='Param1',
#             values='Min'
#         )
#         # 修正：將索引按降序排列，確保圖上最下方是最小值
#         pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
#         min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        
#         xlabel = f"{condition1_name} (%)"
#         ylabel = f"{condition2_name} (%)"
#     else:
#         pivot_table = df.pivot(
#             index='Param1',
#             columns='Param2',
#             values=compare
#         )
#         min_pivot = df.pivot(
#             index='Param1',
#             columns='Param2',
#             values='Min'
#         )
#         pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
#         min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        
#         xlabel = f"{condition2_name} (%)"
#         ylabel = f"{condition1_name} (%)"
    
#     # 設置圖形
#     plt.figure(figsize=figsize)
    
#     # 創建遮罩
#     mask = (min_pivot == 0)
    
#     # 繪製熱力圖
#     sns.heatmap(pivot_table,
#                 annot=True,
#                 annot_kws={'size': 14},
#                 fmt='.2f',
#                 cmap='coolwarm',
#                 cbar_kws={'label': compare},
#                 square=True,
#                 mask=None)
    
#     # 在Min==0的位置上覆蓋統一的灰色方塊
#     for i in range(len(pivot_table.index)):
#         for j in range(len(pivot_table.columns)):
#             if mask.iloc[i, j]:
#                 plt.gca().add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='#808080'))
    
#     # 設置標題和軸標籤
#     if title is None:
#         title = f'{condition1_name} vs {condition2_name} 策略{compare}比較'
#     plt.title(title)
#     plt.xlabel(xlabel)
#     plt.ylabel(ylabel)
    
#     # 調整布局
#     plt.tight_layout()
#     plt.show()


In [ ]:
def plot_strategy_heatmap(df, compare='CAGR (%)', x_first=True, figsize=(14, 6), title=None, 
                         benchmark_param1=None, benchmark_param2=None, rep=None):
    """
    繪製策略熱力圖，當df['Min']==0時顯示灰色，並用白色框線標記Benchmark位置
    如果策略的日期不是從2003年開始，也會顯示灰色

    Parameters:
    -----------
    df : pandas DataFrame
        包含 'Strategy' 和比較欄位的數據框
    compare : str, default='CAGR (%)'
        要比較的欄位名稱，例如 'CAGR (%)' 或 'MDD (%)'
    x_first : bool, default=True
        True: 條件1為X軸，條件2為Y軸
        False: 條件1為Y軸，條件2為X軸
    figsize : tuple, default=(14, 6)
        圖形尺寸
    title : str, optional
        圖表標題，如果不指定則自動生成
    benchmark_param1 : float, optional
        指標1的基準值
    benchmark_param2 : float, optional
        指標2的基準值
    """
    
    if compare not in df.columns:
        raise ValueError(f"Column '{compare}' not found in DataFrame")
    
    param1_values = []
    param2_values = []
    not_start_2003 = []  # 儲存不是從2003年開始的策略
    
    # 從策略名稱中提取參數
    for strategy in df['Strategy']:
        date_index = rep.reports[strategy].position.index
        # print(date_index)
        
        # 檢查是否從2003年開始
        starts_from_2003 = False
        if len(date_index) > 0:
            first_date_str = str(date_index[0])
            if first_date_str.startswith('2003'):
                starts_from_2003 = True
                
        parts = strategy.split('_')
        # 處理可能帶有%的數值
        param1 = float(parts[1].replace('%', ''))
        param2 = float(parts[3].replace('%', ''))
        param1_values.append(param1)
        param2_values.append(param2)
        not_start_2003.append(not starts_from_2003)  # 記錄非2003開始的策略
    
    df['Param1'] = param1_values
    df['Param2'] = param2_values
    df['Not2003Start'] = not_start_2003  # 將結果添加到DataFrame
    
    condition1_name = df['Strategy'].iloc[0].split('_')[0]
    condition2_name = df['Strategy'].iloc[0].split('_')[2]

    if x_first:
        pivot_table = df.pivot(
            index='Param2',
            columns='Param1',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition1_name} (%)"
        ylabel = f"{condition2_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param1)
            benchmark_y = pivot_table.index.get_loc(benchmark_param2)
    else:
        pivot_table = df.pivot(
            index='Param1',
            columns='Param2',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition2_name} (%)"
        ylabel = f"{condition1_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param2)
            benchmark_y = pivot_table.index.get_loc(benchmark_param1)
    
    fig = plt.figure(figsize=figsize)
    
    mask = (min_pivot == 0)
    
    # 繪製熱力圖
    sns.heatmap(pivot_table,
                annot=True,
                annot_kws={'size': 14},
                fmt='.2f',
                cmap='coolwarm',
                cbar_kws={'label': compare},
                square=True,
                mask=None)
    
    # 在Min==0或不是從2003年開始的位置上覆蓋統一的灰色方塊
    for i in range(len(pivot_table.index)):
        for j in range(len(pivot_table.columns)):
            # 如果Min==0或不是從2003年開始，則繪製灰色方塊
            if mask.iloc[i, j] or (not_2003_pivot.iloc[i, j] == True):
                plt.gca().add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='#808080'))
    
    # 如果有指定Benchmark參數，繪製白色框線
    if benchmark_param1 is not None and benchmark_param2 is not None:
        plt.gca().add_patch(plt.Rectangle((benchmark_x, benchmark_y), 1, 1, 
                                        fill=False, edgecolor='white', linewidth=2))
    
    if title is None:
        title = f'{condition1_name} vs {condition2_name} 策略{compare}比較'
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()

    fig.savefig(f'./img/{title}.svg', format='svg', dpi=300)

### ROE & 盈餘再投資率

In [ ]:
roe_rr_opt_base_conds = payout_cond & netprofit_cond & listed_cond

roe_rr_pe_opt_conds = {}

for roevalue in range(10, 41, 5): # ROE5y平均 10~25%
    for rrvalue in range(0, 81, 10):  # 盈再率 0~80%
        rrvalue_opt = rr.copy() < (rrvalue/100)
        roe_5y_opt = roe_rol.copy() > (roevalue/100)
        
        roe_rr_opt_all_conds = (roe_rr_opt_base_conds & roe_5y_opt & rrvalue_opt)[START_DATE:END_DATE]

        roe_rr_pe_opt_conds[f'ROE5年平均_{roevalue}%_盈再率_{rrvalue}%__本益比進出場'] = (roe_rr_opt_all_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~roe_rr_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])



roe_rr_pe_opt_collecs = sim_conditions(roe_rr_pe_opt_conds, resample='M', data=data)
roe_rr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_rr_pe_opt_df = roe_rr_pe_opt_collecs.selected_stock_count_analysis()
roe_rr_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_rr_pe_opt_df,
                        x_first=False, 
                        title='2003~2024 美股_ROE5年平均_盈再率_本益比進出場_CAGR(%)',
                        figsize=(14, 8),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep=roe_rr_pe_opt_collecs)

In [ ]:
fig = roe_rr_pe_opt_collecs.plot_reps_stock_counts(['ROE5年平均_40%_盈再率_20%__本益比進出場', 'ROE5年平均_35%_盈再率_20%__本益比進出場', 'ROE5年平均_25%_盈再率_20%__本益比進出場', 'ROE5年平均_15%_盈再率_40%__本益比進出場'])
fig.savefig(f'./img/圖 76美股策略ROE與盈餘再投資率進場條件變化入選股數比較圖.svg', format='svg', dpi=300)

In [ ]:
plot_strategy_heatmap(roe_rr_pe_opt_df,
                      compare='MDD (%)',
                        x_first=False, 
                        title='2003~2024 美股_ROE5年平均_盈再率_本益比進出場_MDD (%)',
                        figsize=(14, 8),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_rr_pe_opt_collecs)

In [ ]:
roe_rr_pe_opt_collecs.reports['ROE5年平均_25%_盈再率_20%__本益比進出場'].display()

In [ ]:
# plot_strategy_heatmap(roe_rr_pe_opt_df,
#                         compare='MDD (%)',
#                         x_first=False, 
#                         title='2003~2024 台股策略績效分析',
#                         figsize=(18, 8))

### ROE & 配息率

In [ ]:
roe_dpr_opt_base_conds = rr_cond & netprofit_cond & listed_cond

roe_dpr_pe_opt_conds = {}

for roevalue in range(10, 41, 5): # ROE 10~25%
    for povalue in range(0, 56, 5):  # 配息率 0~75%
        dpr_cond_3y_opt = payout_ratio_rol.copy() >= (povalue/100)
        roe_5y_opt = roe_rol.copy() > (roevalue/100)
        
        roe_dpr_opt_all_conds = (roe_dpr_opt_base_conds & roe_5y_opt & dpr_cond_3y_opt)[START_DATE:END_DATE]

        roe_dpr_pe_opt_conds[f'ROE5年平均_{roevalue}%_配息率_{povalue}%__本益比進出場'] = (roe_dpr_opt_all_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~roe_dpr_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])



roe_dpr_pe_opt_collecs = sim_conditions(roe_dpr_pe_opt_conds, resample='M', data=data)
roe_dpr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
# roe_dpr_pe_opt_collecs.reports['ROE5年平均_15%_配息率_15%__本益比進出場'].display()

In [ ]:
roe_dpr_pe_opt_collecs.reports['ROE5年平均_35%_配息率_35%__本益比進出場'].create_stacked_returns_plot()

In [ ]:
roe_dpr_pe_opt_df = roe_dpr_pe_opt_collecs.selected_stock_count_analysis()
roe_dpr_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_dpr_pe_opt_df,
                        x_first=False, 
                        title='2003~2024 美股_ROE5年平均_配息率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_dpr_pe_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_dpr_pe_opt_df,
                        compare='MDD (%)',
                        x_first=False, 
                        title='2003~2024 美股_ROE5年平均_配息率_ROE_MDD (%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep= roe_dpr_pe_opt_collecs)

In [ ]:
fig = roe_dpr_pe_opt_collecs.plot_reps_stock_counts(['ROE5年平均_40%_配息率_45%__本益比進出場', 'ROE5年平均_25%_配息率_45%__本益比進出場', 'ROE5年平均_15%_配息率_40%__本益比進出場'])
fig.savefig(f'./img/圖 79美股策略ROE與股利支付率變化入選股數趨勢比較圖.svg', format='svg', dpi=300)

### ROE & PE

In [ ]:
roe_pee_opt_base_conds = rr_cond & payout_cond & netprofit_cond & listed_cond

roe_pee_opt_conds = {}

for roevalue in range(10, 41, 5): # ROE 10~25%
    for peevalue in range(8, 13, 2):  # 本益比

        roe_5y_opt = roe_rol.copy() > (roevalue/100)
        pee_opt = (pe_daily < peevalue).resample('M').last()
        
        roe_pee_opt_all_conds = (roe_pee_opt_base_conds & roe_5y_opt)[START_DATE:END_DATE]

        roe_pee_opt_conds[f'ROE5年平均_{roevalue}%_本益比_{peevalue}__本益比進出場'] = (roe_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~roe_pee_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])


roe_pee_opt_collecs = sim_conditions(roe_pee_opt_conds, resample='M', data=data)
roe_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
fig = roe_pee_opt_collecs.plot_reps_stock_counts(['ROE5年平均_40%_本益比_10__本益比進出場', 'ROE5年平均_40%_本益比_8__本益比進出場', 'ROE5年平均_25%_本益比_10__本益比進出場', 'ROE5年平均_15%_本益比_12__本益比進出場'])
fig.savefig(f'./img/圖 82美股策略ROE與本益比進場條件變化入選股數比較圖.svg', format='svg', dpi=300)

In [ ]:
roe_pee_opt_df = roe_pee_opt_collecs.selected_stock_count_analysis()
roe_pee_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_pee_opt_df,
                        x_first=1, 
                        title='2003~2024 美股_ROE5年平均_本益比_本益比進出場_CAGR(%)',
                        figsize=(12, 5),
                        benchmark_param1=15,
                        benchmark_param2=12,
                        rep = roe_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_pee_opt_df,
                      compare='MDD (%)',
                        x_first=1, 
                        title='2003~2024 美股_ROE5年平均_本益比_本益比進出場_MDD (%)',
                        figsize=(12, 5),
                        benchmark_param1=15,
                        benchmark_param2=12,
                        rep = roe_pee_opt_collecs)

### 盈再率 & 配息率

In [ ]:
rr_dpr_opt_base_conds = roe_cond & netprofit_cond & listed_cond

rr_dpr_pe_opt_conds = {}

for rrvalue in range(0, 81, 10): # 盈再率 
    for povalue in range(0, 56, 5):  # 配息率 0~55%

        dpr_cond_3y_opt = payout_ratio_rol.copy() >= (povalue/100)
        rrvalue_opt = rr.copy() < (rrvalue/100)
        
        rr_dpr_opt_all_conds = (rr_dpr_opt_base_conds & dpr_cond_3y_opt & rrvalue_opt)[START_DATE:END_DATE]

        rr_dpr_pe_opt_conds[f'盈再率小於_{rrvalue}%_配息率_{povalue}%__本益比進出場'] = (rr_dpr_opt_all_conds & daily_pe_entry[START_DATE:END_DATE]).hold_until((~rr_dpr_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])



rr_dpr_pe_opt_collecs = sim_conditions(rr_dpr_pe_opt_conds, resample='M', data=data)
rr_dpr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
rr_dpr_pe_opt_df = rr_dpr_pe_opt_collecs.selected_stock_count_analysis()
rr_dpr_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(rr_dpr_pe_opt_df,
                        x_first=0, 
                        title='2003~2024 美股_盈再率_配息率_本益比進出場_CAGR(%)',
                        figsize=(14, 8),
                        benchmark_param1=40,
                        benchmark_param2=40,
                        rep = rr_dpr_pe_opt_collecs)

In [ ]:
plot_strategy_heatmap(rr_dpr_pe_opt_df,
                        x_first=0, 
                        compare='MDD (%)',
                        title='2003~2024 美股_盈再率_配息率_本益比進出場_MDD(%)',
                        figsize=(14, 8),
                        benchmark_param1=40,
                        benchmark_param2=40,
                        rep = rr_dpr_pe_opt_collecs)

### 盈再率 & PE

In [ ]:
rr_pee_opt_base_conds = roe_cond & payout_cond & netprofit_cond & listed_cond

rr_pee_opt_conds = {}

for rrvalue in range(0, 81, 10): # 盈再率
    for peevalue in range(8, 13, 2):  # 本益比

        pee_opt = (pe_daily < peevalue).resample('M').last()
        rrvalue_opt = rr.copy() < (rrvalue/100)
        
        rr_pee_opt_all_conds = (rr_pee_opt_base_conds & rrvalue_opt)[START_DATE:END_DATE]

        rr_pee_opt_conds[f'盈再率小於_{rrvalue}%_本益比_{peevalue}__本益比進出場'] = (rr_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~rr_pee_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])

rr_pee_collecs = sim_conditions(rr_pee_opt_conds, resample='M', data=data)
rr_pee_collecs.selected_stock_count_analysis()

In [ ]:
rr_pee_opt_df = rr_pee_collecs.selected_stock_count_analysis()
rr_pee_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(rr_pee_opt_df,
                        x_first=1, 
                        title='2003~2024 美股_盈再率_本益比進場條件_CAGR(%)',
                        figsize=(12, 5),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = rr_pee_collecs)

In [ ]:
plot_strategy_heatmap(rr_pee_opt_df,
                      compare='MDD (%)',
                        x_first=1, 
                        title='2003~2024 美股_盈再率_本益比進場條件_MDD (%)',
                        figsize=(12, 5),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = rr_pee_collecs)

### 配息率 & PE

In [ ]:
dpr_pee_opt_base_conds = roe_cond & rr_cond & netprofit_cond & listed_cond

dpr_pee_opt_conds = {}

for povalue in range(0, 56, 5): # 配息率
    for peevalue in range(8, 13, 2):  # 本益比

        pee_opt = (pe_daily < peevalue).resample('M').last()
        dpr_cond_3y_opt = payout_ratio_rol.copy() >= (povalue/100)
        
        dpr_pee_opt_all_conds = (dpr_pee_opt_base_conds & dpr_cond_3y_opt)[START_DATE:END_DATE]

        dpr_pee_opt_conds[f'配息率_{povalue}%_本益比_{peevalue}__本益比進出場'] = (dpr_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~dpr_pee_opt_all_conds) | daily_pe_exit[START_DATE:END_DATE])

dpr_pee_opt_collecs = sim_conditions(dpr_pee_opt_conds, resample='M', data=data)
dpr_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
dpr_pee_opt_df = dpr_pee_opt_collecs.selected_stock_count_analysis()
dpr_pee_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(dpr_pee_opt_df,
                        x_first=1, 
                        title='2003~2024 美股_配息率_本益比_本益比進出場_CAGR(%)',
                        figsize=(14, 5),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = dpr_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(dpr_pee_opt_df,
                      compare="MDD (%)",
                        x_first=1, 
                        title='2003~2024 美股_配息率_本益比_本益比進出場_MDD (%)',
                        figsize=(14, 5),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = dpr_pee_opt_collecs)